<a href="https://colab.research.google.com/github/cdascientist/Vis/blob/master/vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1: INFRASTRUCTURE — DynamicSection · DynamicProbe · JITObject · TandemCalc · HF · HFEngine · PipelineTelemetry · StoryLog · VarTracker · ScanEventTemporalMatcher · IsomorphicDistributionEngine · JITPipelineArchitecture

from __future__ import annotations
import math
import datetime, os, logging, time, re, threading
from dataclasses import dataclass, field
from typing import Optional, Any


# ── DynamicSection ────────────────────────────────────────────────────────────

class DynamicSection:
    """Generic attribute bag.  __getattr__ returns None, never raises."""
    def __init__(self): self._data = {}
    def __getattr__(self, k): return self._data.get(k)
    def __setattr__(self, k, v):
        if k == '_data': super().__setattr__(k, v)
        else: self._data[k] = v
    def to_dict(self):
        return {k: (v.to_dict() if isinstance(v, DynamicSection) else v)
                for k, v in self._data.items()}


# ── DynamicProbe ──────────────────────────────────────────────────────────────

@dataclass
class DynamicProbe:
    uid:       str            = ""
    payload:   DynamicSection = field(default_factory=DynamicSection)
    telemetry: Optional[Any]  = None
    def __post_init__(self):
        self.payload.stat_1 = 1; self.payload.stat_2 = 2; self.payload.stat_3 = 3
        self.payload.stat_4 = 4; self.payload.stat_5 = 5


# ── JITObject ─────────────────────────────────────────────────────────────────

@dataclass
class JITObject(DynamicProbe):
    jit_metrics:      DynamicSection = field(default_factory=DynamicSection)
    tandem_interface: DynamicSection = field(default_factory=DynamicSection)   # THFPI SHARED BUS — never rename

    def __post_init__(self):
        super().__post_init__()
        if not self.uid: self.uid = "JIT-CORE-ENGINE"

        self.jit_metrics.embedded_results = DynamicSection()
        self.tandem_interface.results     = DynamicSection()

        # ── THFPI CENTROID STORE ──────────────────────────────────────────────
        self.tandem_interface.kmeans_centroids = DynamicSection()

        # ── THFPI VELOCITY SUB-COMPONENT ─────────────────────────────────────
        self.tandem_interface.synthetic_centroid_velocity_subcomponent = DynamicSection()

        # ── ISOMORPHIC DISTRIBUTION ENGINE ───────────────────────────────────
        self.tandem_interface.iso_central = DynamicSection()
        for _n in range(8):
            setattr(self.tandem_interface, f"iso_node_{_n}", DynamicSection())

        # ── ISO TELEMETRY LOG ─────────────────────────────────────────────────
        self.tandem_interface.iso_telemetry_log = []

    def evaluate_regex(self, pattern, text):
        matches = re.findall(pattern, str(text))
        return float(matches[0]) if matches else 0.0
    def compute_addition(self, a, b):    return float(a or 0) + float(b or 0)
    def compute_subtraction(self, a, b): return float(a or 0) - float(b or 0)
    def signature(self, data):           return f"SEAL_HASH_{data}"


# ── TandemCalc — THFPI EMBEDDED FUNCTION LIBRARY ─────────────────────────────

class TandemCalc:

    @staticmethod
    def manual_phase_calc(a, b):   return float(a or 0) * float(b or 0)        # [TC-01]
    @staticmethod
    def checkpoint_entry(a, b):    return f"ENTRY_SYNC_[{a}]_[{b}]"            # [TC-02]
    @staticmethod
    def checkpoint_between(a, b):  return f"TRANSIT_SYNC_[{a}]_[{b}]"         # [TC-03]
    @staticmethod
    def checkpoint_exit(a, b):     return f"EXIT_SYNC_[{a}]_[{b}]"            # [TC-04]
    @staticmethod
    def subtract(a, b):            return float(a or 0) - float(b or 0)        # [TC-05]

    @staticmethod
    def normalize_to_day_fraction(ts_primary_utc_string, ts_secondary_utc_string):
        from datetime import datetime as _dt
        _fmt = "%Y-%m-%d %H:%M:%S"                                             # [TC-N1]
        _day = 86400.0                                                          # [TC-N2]
        ep   = _dt.strptime(ts_primary_utc_string,   _fmt).timestamp()         # [TC-N3]
        es   = _dt.strptime(ts_secondary_utc_string, _fmt).timestamp()         # [TC-N4]
        pri  = _dt.strptime(ts_primary_utc_string,   _fmt)                     # [TC-N5]
        anc  = pri.strftime("%Y-%m-%d") + " 00:00:00"                          # [TC-N6]
        ea   = _dt.strptime(anc, _fmt).timestamp()                             # [TC-N7]
        np_  = (ep - ea) / _day                                                # [TC-N8]
        ns   = (es - ea) / _day                                                # [TC-N9]
        return {"norm_primary": np_, "norm_secondary": ns,                     # [TC-N11]
                "signed_delta": np_ - ns, "anchor_utc_string": anc, "anchor_epoch": ea}

    @staticmethod
    def kmeans_three_centroid_synthetic(norm_primary, norm_secondary):
        _max = 100                                                              # [TC-K1]
        syn  = (norm_primary + norm_secondary) / 2.0                           # [TC-K2]
        c0, c1, c2 = norm_primary, syn, norm_secondary                         # [TC-K3]
        iters = 0                                                               # [TC-K4]
        for _ in range(_max):                                                   # [TC-K5]
            d0 = [v for v in [norm_primary, norm_secondary]
                  if abs(v-c0) <= abs(v-c1) and abs(v-c0) <= abs(v-c2)]
            d1 = [v for v in [norm_primary, norm_secondary]
                  if abs(v-c1) <  abs(v-c0) and abs(v-c1) <= abs(v-c2)]
            d2 = [v for v in [norm_primary, norm_secondary]
                  if abs(v-c2) <  abs(v-c0) and abs(v-c2) <  abs(v-c1)]
            n0 = sum(d0)/len(d0) if d0 else c0
            n1 = sum(d1)/len(d1) if d1 else c1
            n2 = sum(d2)/len(d2) if d2 else c2
            if n0 == c0 and n1 == c1 and n2 == c2: break
            c0, c1, c2 = n0, n1, n2; iters += 1
        s = sorted([c0, c1, c2])
        return {"centroid_0_primary": s[0], "centroid_1_synthetic_midpoint": s[1],
                "centroid_2_secondary": s[2], "synthetic_seed": syn, "iteration_count": iters}

    @staticmethod
    def conjunctive_median_delta(c0, c1_syn, c2):
        d01 = c1_syn - c0                                                       # [TC-C1]
        d12 = c2 - c1_syn                                                       # [TC-C2]
        s   = sorted([d01, d12])
        med = (s[0] + s[1]) / 2.0                                              # [TC-C3]
        return {"diff_0_to_1": d01, "diff_1_to_2": d12,
                "conjunctive_median": med, "extrapolated_delta": med}

    @staticmethod
    def synthetic_centroid_tension_velocity(
            synthetic_centroid_day_fraction_position,
            list_of_all_real_centroid_day_fraction_positions):

        from decimal import Decimal, getcontext
        getcontext().prec = 50                                                 # [TC-V-00]
        _syn = synthetic_centroid_day_fraction_position                        # [TC-V-01]
        _n   = len(list_of_all_real_centroid_day_fraction_positions)           # [TC-V-02]
        per_centroid_signed_tension_vector_list = [
            _c - _syn for _c in list_of_all_real_centroid_day_fraction_positions]
        per_centroid_squared_tension_value_list = [
            _t ** 2 for _t in per_centroid_signed_tension_vector_list]
        _sum_sq = sum(per_centroid_squared_tension_value_list)
        cumulative_magnitude_velocity_in_day_fractions = math.sqrt(_sum_sq) if _sum_sq > 0.0 else 0.0
        cumulative_magnitude_velocity_in_minutes = cumulative_magnitude_velocity_in_day_fractions * 1440.0
        tension_symmetry_indicator = sum(per_centroid_signed_tension_vector_list)
        mean_absolute_tension = (sum(abs(_t) for _t in per_centroid_signed_tension_vector_list) / _n) if _n > 0 else 0.0
        velocity_normalized = (cumulative_magnitude_velocity_in_day_fractions / mean_absolute_tension
                                if mean_absolute_tension > 0.0 else 0.0)
        velocity_normalized_sqrt_n_reference = math.sqrt(_n) if _n > 0 else 0.0
        velocity_normalized_deviation_from_sqrt_n = velocity_normalized - velocity_normalized_sqrt_n_reference
        _D_real = [Decimal(repr(_c)) for _c in list_of_all_real_centroid_day_fraction_positions]
        _D_syn  = sum(_D_real) / Decimal(_n)
        _D_tensions = [_c - _D_syn for _c in _D_real]
        _D_sum_sq   = sum(_t ** 2 for _t in _D_tensions)
        _D_velocity = _D_sum_sq.sqrt() if _D_sum_sq > 0 else Decimal(0)
        _D_vel_min  = _D_velocity * Decimal(1440)
        _D_symmetry = sum(_D_tensions)
        _D_mean_abs = sum(abs(_t) for _t in _D_tensions) / Decimal(_n)
        _D_normalized = (_D_velocity / _D_mean_abs if _D_mean_abs > 0 else Decimal(0))
        _D_sqrt_n   = Decimal(_n).sqrt()
        _D_norm_dev = _D_normalized - _D_sqrt_n
        return {
            "synthetic_centroid_position":               _syn,
            "real_centroid_count":                       _n,
            "per_centroid_signed_tension_vectors":       per_centroid_signed_tension_vector_list,
            "per_centroid_squared_tension_values":       per_centroid_squared_tension_value_list,
            "sum_of_all_squared_tensions":               _sum_sq,
            "cumulative_magnitude_velocity":             cumulative_magnitude_velocity_in_day_fractions,  # [TC-V-OUT-06] ISO period driver
            "velocity_in_minutes":                       cumulative_magnitude_velocity_in_minutes,
            "tension_symmetry_indicator":                tension_symmetry_indicator,
            "mean_absolute_tension":                     mean_absolute_tension,
            "velocity_normalized":                       velocity_normalized,                       # [TC-V-OUT-10] ISO C(t) driver
            "velocity_normalized_sqrt_n_reference":      velocity_normalized_sqrt_n_reference,
            "velocity_normalized_deviation_from_sqrt_n": velocity_normalized_deviation_from_sqrt_n,
            "decimal_velocity_50fig":                    str(_D_velocity),
            "decimal_velocity_minutes_50fig":            str(_D_vel_min),
            "decimal_symmetry_indicator_50fig":          str(_D_symmetry),
            "decimal_normalized_50fig":                  str(_D_normalized),
            "decimal_normalized_deviation_50fig":        str(_D_norm_dev),
        }

    @staticmethod
    def _pipeline_body(ts_pri, ts_sec, tag):
        norm   = TandemCalc.normalize_to_day_fraction(ts_pri, ts_sec)
        kmeans = TandemCalc.kmeans_three_centroid_synthetic(norm["norm_primary"], norm["norm_secondary"])
        delta  = TandemCalc.conjunctive_median_delta(
                     kmeans["centroid_0_primary"],
                     kmeans["centroid_1_synthetic_midpoint"],
                     kmeans["centroid_2_secondary"])
        return {
            "norm_primary":                         norm["norm_primary"],
            "norm_secondary":                       norm["norm_secondary"],
            "norm_signed_delta":                    norm["signed_delta"],
            "norm_anchor_utc_string":               norm["anchor_utc_string"],
            "norm_anchor_epoch":                    norm["anchor_epoch"],
            "kmeans_centroid_0_primary":            kmeans["centroid_0_primary"],
            "kmeans_centroid_1_synthetic_midpoint": kmeans["centroid_1_synthetic_midpoint"],
            "kmeans_centroid_2_secondary":          kmeans["centroid_2_secondary"],
            "kmeans_synthetic_seed":                kmeans["synthetic_seed"],
            "kmeans_iteration_count":               kmeans["iteration_count"],
            "delta_diff_0_to_1":                    delta["diff_0_to_1"],
            "delta_diff_1_to_2":                    delta["diff_1_to_2"],
            "delta_conjunctive_median":             delta["conjunctive_median"],
            "delta_extrapolated":                   delta["extrapolated_delta"],
        }

    @staticmethod
    def thfpi_group1_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P1")  # [TC-P1]
    @staticmethod
    def thfpi_group2_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P2")  # [TC-P2]
    @staticmethod
    def thfpi_group3_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P3")  # [TC-P3]
    @staticmethod
    def thfpi_group4_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P4")  # [TC-P4]

    @staticmethod
    def thfpi_probe_pair_utc_timestamps_to_conjunctive_median_delta_via_day_normalized_synthetic_midpoint_seeded_kmeans_with_synthetic_centroid_tension_velocity(
            primary_probe_scan_event_utc_timestamp_string,
            secondary_probe_scan_event_utc_timestamp_string):

        from datetime import datetime as _dt
        _fmt = "%Y-%m-%d %H:%M:%S"
        _day = 86400.0
        _ep  = _dt.strptime(primary_probe_scan_event_utc_timestamp_string,   _fmt).timestamp()
        _es  = _dt.strptime(secondary_probe_scan_event_utc_timestamp_string, _fmt).timestamp()
        _pri = _dt.strptime(primary_probe_scan_event_utc_timestamp_string,   _fmt)
        _anc_str = _pri.strftime("%Y-%m-%d") + " 00:00:00"
        _ea  = _dt.strptime(_anc_str, _fmt).timestamp()
        stage_1_norm_primary   = (_ep - _ea) / _day
        stage_1_norm_secondary = (_es - _ea) / _day
        stage_1_signed_delta   = stage_1_norm_primary - stage_1_norm_secondary

        _max = 100
        stage_2_synthetic_seed = (stage_1_norm_primary + stage_1_norm_secondary) / 2.0
        _c0, _c1, _c2 = stage_1_norm_primary, stage_2_synthetic_seed, stage_1_norm_secondary
        _iters = 0
        for _ in range(_max):
            _d0 = [v for v in [stage_1_norm_primary, stage_1_norm_secondary]
                   if abs(v-_c0) <= abs(v-_c1) and abs(v-_c0) <= abs(v-_c2)]
            _d1 = [v for v in [stage_1_norm_primary, stage_1_norm_secondary]
                   if abs(v-_c1) <  abs(v-_c0) and abs(v-_c1) <= abs(v-_c2)]
            _d2 = [v for v in [stage_1_norm_primary, stage_1_norm_secondary]
                   if abs(v-_c2) <  abs(v-_c0) and abs(v-_c2) <  abs(v-_c1)]
            _n0 = sum(_d0)/len(_d0) if _d0 else _c0
            _n1 = sum(_d1)/len(_d1) if _d1 else _c1
            _n2 = sum(_d2)/len(_d2) if _d2 else _c2
            if _n0 == _c0 and _n1 == _c1 and _n2 == _c2: break
            _c0, _c1, _c2 = _n0, _n1, _n2; _iters += 1
        _sorted = sorted([_c0, _c1, _c2])
        stage_2_c0 = _sorted[0]; stage_2_c1 = _sorted[1]; stage_2_c2 = _sorted[2]
        stage_2_iteration_count = _iters

        stage_3_diff_0_to_1 = stage_2_c1 - stage_2_c0
        stage_3_diff_1_to_2 = stage_2_c2 - stage_2_c1
        _s3 = sorted([stage_3_diff_0_to_1, stage_3_diff_1_to_2])
        stage_3_conjunctive_median = (_s3[0] + _s3[1]) / 2.0

        _tc_v_result = TandemCalc.synthetic_centroid_tension_velocity(stage_2_c1, [stage_2_c0, stage_2_c2])
        stage_4_synthetic_position       = _tc_v_result["synthetic_centroid_position"]
        stage_4_real_centroid_count      = _tc_v_result["real_centroid_count"]
        stage_4_signed_tensions          = _tc_v_result["per_centroid_signed_tension_vectors"]
        stage_4_squared_tensions         = _tc_v_result["per_centroid_squared_tension_values"]
        stage_4_sum_squared              = _tc_v_result["sum_of_all_squared_tensions"]
        stage_4_velocity_day_frac        = _tc_v_result["cumulative_magnitude_velocity"]
        stage_4_velocity_minutes         = _tc_v_result["velocity_in_minutes"]
        stage_4_symmetry_indicator       = _tc_v_result["tension_symmetry_indicator"]
        stage_4_mean_abs_tension         = _tc_v_result["mean_absolute_tension"]
        stage_4_velocity_normalized      = _tc_v_result["velocity_normalized"]
        stage_4_velocity_norm_sqrt_n_ref = _tc_v_result["velocity_normalized_sqrt_n_reference"]
        stage_4_velocity_norm_deviation  = _tc_v_result["velocity_normalized_deviation_from_sqrt_n"]
        stage_4_decimal_velocity_50fig   = _tc_v_result["decimal_velocity_50fig"]
        stage_4_decimal_vel_min_50fig    = _tc_v_result["decimal_velocity_minutes_50fig"]
        stage_4_decimal_symmetry_50fig   = _tc_v_result["decimal_symmetry_indicator_50fig"]
        stage_4_decimal_normalized_50fig = _tc_v_result["decimal_normalized_50fig"]
        stage_4_decimal_norm_dev_50fig   = _tc_v_result["decimal_normalized_deviation_50fig"]

        return {
            "norm_primary":                               stage_1_norm_primary,
            "norm_secondary":                             stage_1_norm_secondary,
            "norm_signed_delta":                          stage_1_signed_delta,
            "norm_anchor_utc_string":                     _anc_str,
            "norm_anchor_epoch":                          _ea,
            "kmeans_centroid_0_primary":                  stage_2_c0,
            "kmeans_centroid_1_synthetic_midpoint":       stage_2_c1,
            "kmeans_centroid_2_secondary":                stage_2_c2,
            "kmeans_synthetic_seed":                      stage_2_synthetic_seed,
            "kmeans_iteration_count":                     stage_2_iteration_count,
            "delta_diff_0_to_1":                          stage_3_diff_0_to_1,
            "delta_diff_1_to_2":                          stage_3_diff_1_to_2,
            "delta_conjunctive_median":                   stage_3_conjunctive_median,
            "delta_extrapolated":                         stage_3_conjunctive_median,
            "velocity_synthetic_position":                stage_4_synthetic_position,
            "velocity_real_centroid_count":               stage_4_real_centroid_count,
            "velocity_per_centroid_signed_tensions":      stage_4_signed_tensions,
            "velocity_per_centroid_squared_tensions":     stage_4_squared_tensions,
            "velocity_sum_of_squared_tensions":           stage_4_sum_squared,
            "velocity_cumulative_magnitude":              stage_4_velocity_day_frac,     # ISO period driver
            "velocity_in_minutes":                        stage_4_velocity_minutes,
            "velocity_tension_symmetry_indicator":        stage_4_symmetry_indicator,
            "velocity_mean_absolute_tension":             stage_4_mean_abs_tension,
            "velocity_normalized":                        stage_4_velocity_normalized,   # ISO C(t) driver
            "velocity_normalized_sqrt_n_reference":       stage_4_velocity_norm_sqrt_n_ref,
            "velocity_normalized_deviation_from_sqrt_n":  stage_4_velocity_norm_deviation,
            "velocity_decimal_50fig":                     stage_4_decimal_velocity_50fig,
            "velocity_decimal_minutes_50fig":             stage_4_decimal_vel_min_50fig,
            "velocity_decimal_symmetry_50fig":            stage_4_decimal_symmetry_50fig,
            "velocity_decimal_normalized_50fig":          stage_4_decimal_normalized_50fig,
            "velocity_decimal_norm_deviation_50fig":      stage_4_decimal_norm_dev_50fig,
        }


# ═══════════════════════════════════════════════════════════════════════════════
# IsomorphicDistributionEngine   NOVEL
# ═══════════════════════════════════════════════════════════════════════════════
#
# Stateless — no class-level timer. elapsed_seconds computed by Orchestrator
# as (perf_counter_ns() − tel.start_ns) / 1e9 and passed as explicit argument.
#
# C(t)   = ISO_M · velocity_normalized + ISO_B          (HF TC-V scalar → magnitude)
# period = ISO_BASE_PERIOD · cumulative_magnitude        (HF TC-V scalar → rate)
# t      = triangle_wave(elapsed_seconds, period)        (phase clock only)
# w_i    = 1 + sin(2π·i/N + π/4)                        (asymmetric sine weights)
# V_i    = C · w_i / N_connected                        (throughput-conserving)
# P_i'   = P_i · (1 + V_i · 0.5)                        (isometric outward push)
# hue    = V_i · 180°                                    (HSL colour mapping)

class IsomorphicDistributionEngine:

    ISO_M                   = 1.50   # [ISO-CONST-01]
    ISO_B                   = 0.20   # [ISO-CONST-02]
    ISO_BASE_PERIOD_SECONDS = 4.0    # [ISO-CONST-03]
    ISO_MIN_PERIOD_SCALE    = 1e-6   # [ISO-CONST-04]

    ISO_VERTEX_BASE_POSITIONS = [    # [ISO-CONST-05]
        [ 1.0,  1.0,  1.0],   # node 0
        [ 1.0,  1.0, -1.0],   # node 1
        [ 1.0, -1.0,  1.0],   # node 2
        [ 1.0, -1.0, -1.0],   # node 3
        [-1.0,  1.0,  1.0],   # node 4
        [-1.0,  1.0, -1.0],   # node 5
        [-1.0, -1.0,  1.0],   # node 6
        [-1.0, -1.0, -1.0],   # node 7
    ]

    @staticmethod
    def triangle_wave(elapsed_seconds, period_seconds):                        # [ISO-01]
        _period = max(float(period_seconds), IsomorphicDistributionEngine.ISO_MIN_PERIOD_SCALE)
        _phase  = (float(elapsed_seconds) % _period) / _period
        return 1.0 - 2.0 * abs(_phase - 0.5)

    @staticmethod
    def central_node_value(velocity_normalized):                               # [ISO-02]
        return (IsomorphicDistributionEngine.ISO_M * float(velocity_normalized)
                + IsomorphicDistributionEngine.ISO_B)

    @staticmethod
    def vertex_weights(n_connected):                                           # [ISO-03]
        _n = max(int(n_connected), 1)
        return [1.0 + math.sin(2.0 * math.pi * _i / _n + math.pi / 4.0) for _i in range(_n)]

    @staticmethod
    def distributed_value(central_value_C, weight_w_i, n_connected):          # [ISO-04]
        return float(central_value_C) * float(weight_w_i) / max(int(n_connected), 1)

    @staticmethod
    def updated_position(base_position_p_i, distributed_value_v_i):           # [ISO-05]
        _scale = 1.0 + float(distributed_value_v_i) * 0.5
        return [_coord * _scale for _coord in base_position_p_i]

    @staticmethod
    def vertex_hue(distributed_value_v_i):                                     # [ISO-06]
        return float(distributed_value_v_i) * 180.0

    @staticmethod
    def vertex_scale(distributed_value_v_i):                                   # [ISO-07]
        return 1.0 + float(distributed_value_v_i) * 0.5

    @staticmethod
    def sync(elapsed_seconds, velocity_normalized, cumulative_magnitude,
             jit, matched_groups, sync_count):                                 # [ISO-08]
        IDE = IsomorphicDistributionEngine

        _cum_mag        = max(float(cumulative_magnitude), IDE.ISO_MIN_PERIOD_SCALE)   # [ISO-08-01]
        _period_seconds = IDE.ISO_BASE_PERIOD_SECONDS * _cum_mag                       # [ISO-08-02]
        _t              = IDE.triangle_wave(float(elapsed_seconds), _period_seconds)   # [ISO-08-03]
        _vel_norm       = float(velocity_normalized)                                   # [ISO-08-04]
        _C              = IDE.central_node_value(_vel_norm)                            # [ISO-08-05]
        _N              = len(IDE.ISO_VERTEX_BASE_POSITIONS)                           # [ISO-08-06]
        _weights        = IDE.vertex_weights(_N)                                       # [ISO-08-07]

        jit.tandem_interface.iso_central.sync_count                  = int(sync_count)         # [ISO-08-08]
        jit.tandem_interface.iso_central.elapsed_seconds             = float(elapsed_seconds)  # [ISO-08-09]
        jit.tandem_interface.iso_central.period_seconds              = _period_seconds         # [ISO-08-10]
        jit.tandem_interface.iso_central.triangle_wave_t             = _t                      # [ISO-08-11]
        jit.tandem_interface.iso_central.velocity_normalized_scalar  = _vel_norm               # [ISO-08-12]
        jit.tandem_interface.iso_central.cumulative_magnitude_scalar = float(cumulative_magnitude)  # [ISO-08-13]
        jit.tandem_interface.iso_central.central_value_C             = _C                      # [ISO-08-14]
        jit.tandem_interface.iso_central.iso_m                       = IDE.ISO_M               # [ISO-08-15]
        jit.tandem_interface.iso_central.iso_b                       = IDE.ISO_B               # [ISO-08-16]
        jit.tandem_interface.iso_central.n_connected                 = _N                      # [ISO-08-17]
        jit.tandem_interface.iso_central.formula_string              = (                       # [ISO-08-18]
            f"C = {IDE.ISO_M}·{_vel_norm:.6f} + {IDE.ISO_B} = {_C:.6f}  "
            f"t={_t:.6f}  period={_period_seconds:.6f}s  "
            f"elapsed={float(elapsed_seconds):.6f}s  N={_N}"
        )

        _node_snapshots = []
        for _i in range(_N):                                                   # [ISO-08-20]
            _w     = _weights[_i]                                              # [ISO-08-21]
            _P     = IDE.ISO_VERTEX_BASE_POSITIONS[_i]                         # [ISO-08-22]
            _V     = IDE.distributed_value(_C, _w, _N)                        # [ISO-08-23]
            _Pp    = IDE.updated_position(_P, _V)                              # [ISO-08-24]
            _scale = IDE.vertex_scale(_V)                                      # [ISO-08-25]
            _hue   = IDE.vertex_hue(_V)                                        # [ISO-08-26]
            _mg    = matched_groups[_i] if _i < len(matched_groups) else {}    # [ISO-08-27]

            _node  = getattr(jit.tandem_interface, f"iso_node_{_i}")           # [ISO-08-28]
            _node.node_index                        = _i                       # [ISO-08-29]
            _node.weight_w_i                        = _w                       # [ISO-08-30]
            _node.base_position_x                   = _P[0]                    # [ISO-08-31]
            _node.base_position_y                   = _P[1]                    # [ISO-08-32]
            _node.base_position_z                   = _P[2]                    # [ISO-08-33]
            _node.distributed_value_V_i             = _V                       # [ISO-08-34]
            _node.updated_position_x_prime          = _Pp[0]                   # [ISO-08-35]
            _node.updated_position_y_prime          = _Pp[1]                   # [ISO-08-36]
            _node.updated_position_z_prime          = _Pp[2]                   # [ISO-08-37]
            _node.scale_factor                      = _scale                   # [ISO-08-38]
            _node.hue_degrees                       = _hue                     # [ISO-08-39]
            _node.matched_group_sequential_number   = _mg.get("matched_group_sequential_number",   _i + 1)  # [ISO-08-40]
            _node.matched_group_pri_location        = _mg.get("matched_group_primary_probe_contributing_scan_event",   {}).get("scan_event_physical_location_city_state", "")  # [ISO-08-41]
            _node.matched_group_sec_location        = _mg.get("matched_group_secondary_probe_contributing_scan_event", {}).get("scan_event_physical_location_city_state", "")  # [ISO-08-42]
            _node.matched_group_time_gap_hours      = _mg.get("matched_group_actual_time_gap_between_probe_events_hours", 0.0)  # [ISO-08-43]
            _node.formula_string                    = (                        # [ISO-08-44]
                f"w_{_i}={_w:.4f}  V_{_i}=C·{_w:.4f}/{_N}={_V:.4f}  "
                f"scale={_scale:.4f}  hue={_hue:.1f}°  "
                f"P'=[{_Pp[0]:.4f},{_Pp[1]:.4f},{_Pp[2]:.4f}]"
            )
            _node_snapshots.append({                                           # [ISO-08-45]
                "node_index": _i, "weight_w_i": _w, "distributed_value_V_i": _V,
                "scale_factor": _scale, "hue_degrees": _hue, "updated_position": _Pp,
            })

        jit.tandem_interface.iso_telemetry_log.append({                        # [ISO-08-46]
            "sync_count": int(sync_count), "elapsed_seconds": float(elapsed_seconds),
            "period_seconds": _period_seconds, "triangle_wave_t": _t,
            "velocity_normalized_scalar": _vel_norm,
            "cumulative_magnitude_scalar": float(cumulative_magnitude),
            "central_value_C": _C, "n_connected": _N,
            "node_snapshots": _node_snapshots,
        })
        return jit.tandem_interface.iso_central                                # [ISO-08-47]

    @staticmethod
    def analyse_cumulative_magnitude_telemetry(jit):                           # [ISO-09]
        _log = jit.tandem_interface.iso_telemetry_log or []
        if not _log: return {}
        _all_C        = [e["central_value_C"]             for e in _log]
        _all_vel_norm = [e["velocity_normalized_scalar"]   for e in _log]
        _all_cum_mag  = [e["cumulative_magnitude_scalar"]  for e in _log]
        _all_t        = [e["triangle_wave_t"]              for e in _log]
        _all_period   = [e["period_seconds"]               for e in _log]
        _n_nodes      = len(IsomorphicDistributionEngine.ISO_VERTEX_BASE_POSITIONS)
        _v_envelope   = {}
        for _i in range(_n_nodes):
            _all_V_i = [e["node_snapshots"][_i]["distributed_value_V_i"]
                        for e in _log if _i < len(e["node_snapshots"])]
            _v_envelope[f"node_{_i}"] = {
                "V_min":  min(_all_V_i) if _all_V_i else 0.0,
                "V_max":  max(_all_V_i) if _all_V_i else 0.0,
                "V_mean": sum(_all_V_i) / len(_all_V_i) if _all_V_i else 0.0,
            }
        _result = {
            "total_sync_cycles":        len(_log),
            "elapsed_span_seconds":     (_log[-1]["elapsed_seconds"] - _log[0]["elapsed_seconds"]) if len(_log) > 1 else 0.0,
            "C_min": min(_all_C), "C_max": max(_all_C), "C_mean": sum(_all_C)/len(_all_C),
            "velocity_normalized_min":  min(_all_vel_norm),
            "velocity_normalized_max":  max(_all_vel_norm),
            "cumulative_magnitude_min": min(_all_cum_mag),
            "cumulative_magnitude_max": max(_all_cum_mag),
            "period_seconds_min":       min(_all_period),
            "period_seconds_max":       max(_all_period),
            "triangle_wave_t_min":      min(_all_t),
            "triangle_wave_t_max":      max(_all_t),
            "per_node_V_i_envelope":    _v_envelope,
        }
        _analysis = DynamicSection()
        for _k, _v in _result.items():
            if isinstance(_v, dict):
                _sub = DynamicSection()
                for _sk, _sv in _v.items(): setattr(_sub, _sk, _sv)
                setattr(_analysis, _k, _sub)
            else:
                setattr(_analysis, _k, _v)
        jit.tandem_interface.iso_telemetry_analysis = _analysis
        return _result


# ── HF ROUTING TABLE ──────────────────────────────────────────────────────────

_TC_PIPE = TandemCalc.thfpi_probe_pair_utc_timestamps_to_conjunctive_median_delta_via_day_normalized_synthetic_midpoint_seeded_kmeans_with_synthetic_centroid_tension_velocity

HF = {
    "Phase_1": {
        "on_entry": [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_1","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_1"}],
        "on_exit":  [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_1","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_1"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_1","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit checkpoint"},          # [HF-T1]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.thfpi_group1","fn":TandemCalc.thfpi_group1_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 1 TC-N TC-K TC-C"},  # [HF-T2]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.thfpi_group1_consolidated","fn":_TC_PIPE,"op_str":"THFPI_CONSOLIDATED_WITH_VELOCITY","note":"THFPI Group 1 TC-PIPE S1+S2+S3+S4"},  # [HF-T3]
        ],
    },
    "Phase_2": {
        "on_entry":   [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_2","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_2"}],
        "on_exit":    [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_2","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_2"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_2","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit scan_event_2"},
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.thfpi_group2","fn":TandemCalc.thfpi_group2_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 2"},
        ],
    },
    "Phase_3": {
        "on_entry":   [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_3","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_3"}],
        "on_exit":    [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_3","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_3"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_3","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit scan_event_3"},
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.thfpi_group3","fn":TandemCalc.thfpi_group3_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 3"},
        ],
    },
    "Phase_4": {
        "on_entry":   [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_4","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_4"}],
        "on_exit":    [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_4","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_4"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_4","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit scan_event_4"},
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.thfpi_group4","fn":TandemCalc.thfpi_group4_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 4"},
        ],
    },
}


# ── HFEngine ──────────────────────────────────────────────────────────────────

class HFEngine:
    last_retrieval = {}

    @staticmethod
    def _get(obj, path):
        for p in path.split("."): obj = getattr(obj, p, None)
        return obj

    @staticmethod
    def _set(obj, path, value):
        parts = path.split(".")
        for p in parts[:-1]: obj = getattr(obj, p, None)
        if obj: setattr(obj, parts[-1], value)

    @staticmethod
    def fire(node, moment, jit, pri, sec, tel):
        HFEngine.last_retrieval = {"node": node, "moment": moment, "reads": []}
        for r in HF.get(node, {}).get(moment, []):
            if r["direction"] == "merge_to_jit":
                vp  = HFEngine._get(pri, r["src_pri"])
                vs  = HFEngine._get(sec, r["src_sec"])
                res = r["fn"](vp, vs) if r.get("fn") else (vp, vs)
                HFEngine._set(jit, r["dst"], res)
                op   = r.get("op_str", "OP")
                expr = f"{r['src_pri']}({vp}) {op} {r['src_sec']}({vs})"
                tel.record_hf(node, moment, "merge_to_jit", r["dst"], res)
                HFEngine.last_retrieval["reads"].append({
                    "rule_note":    r.get("note", ""),
                    "src_pri_path": r["src_pri"],  "vp_retrieved": vp,
                    "src_sec_path": r["src_sec"],  "vs_retrieved": vs,
                    "fn_called":    r["fn"].__name__ if r.get("fn") else "none",
                    "result":       res, "dst_path": r["dst"], "expression": expr,
                })


# ── PipelineTelemetry ─────────────────────────────────────────────────────────

class PipelineTelemetry:
    def __init__(self):
        self.start_ns = time.perf_counter_ns(); self.uid = "TEL-CORE"
        self.entries = {}; self.durs = {}; self.skews = {}
        self.impacts = {}; self.hf_log = []; self._lock = threading.Lock()
        self.hf_event_log = []

    def enter(self, name):
        with self._lock: self.entries[name] = time.perf_counter_ns(); return f"ENT-{name}"
    def exit(self, name):
        with self._lock: self.durs[name] = time.perf_counter_ns() - self.entries.get(name, 0); return f"EXT-{name}"
    def record_hf(self, p, m, d, dst, v):
        with self._lock: self.hf_log.append((p, m, d, dst, str(v)[:120]))
    def record_hf_event(self, node, moment, reads):
        with self._lock:
            self.hf_event_log.append({
                "node": node, "moment": moment, "n_rules": len(reads),
                "fn_names":  [r.get("fn_called", "?") for r in reads],
                "dst_paths": [r.get("dst_path",  "?") for r in reads],
                "ts_ns": time.perf_counter_ns(),
            })
    def record_metrics(self, phase, skew, impact):
        with self._lock: self.skews[phase] = skew; self.impacts[phase] = impact


# ── StoryLog ──────────────────────────────────────────────────────────────────

class StoryLog:
    _buffer = []; _lock = threading.Lock()

    @classmethod
    def trace(cls, phase, narrative, ops, metrics, jit, pri, sec, tel):
        with cls._lock:
            cls._buffer.append({
                "phase": phase, "narrative": narrative,
                "ops": ops, "metrics": metrics,
                "ts_ns": time.perf_counter_ns(),
            })

    @classmethod
    def flush_buffer(cls):
        with cls._lock:
            out = list(cls._buffer); cls._buffer.clear()
        return out

    @classmethod
    def dump_objects(cls, *objs):
        for o in objs: print(o)


# ── VarTracker ────────────────────────────────────────────────────────────────

class VarTracker:
    _log = []; _step = 0; DIVIDER = "─" * 76

    @classmethod
    def snapshot(cls, label, variables: dict):
        cls._step += 1
        entry = {"step": cls._step, "label": label, "ts_ns": time.perf_counter_ns(), "vars": {}}
        lines = [f"\n{'━'*76}", f"  ◉ BP{cls._step:02d} │ {label}", cls.DIVIDER]
        for name, val in variables.items():
            display = val.to_dict() if isinstance(val, DynamicSection) else \
                      {k: v for k, v in val.__dict__.items() if not k.startswith('_')} \
                      if hasattr(val, '__dict__') and not isinstance(val, type) else val
            entry["vars"][name] = display
            lines.append(f"  {name:<68} = {repr(display)}")
        lines.append(cls.DIVIDER)
        cls._log.append(entry); print("\n".join(lines))

    @classmethod
    def delta(cls, label, variables: dict):
        cls.snapshot(label, variables)

    @classmethod
    def summary(cls):
        print(f"\n{'█'*76}\n  VARIABLE TRACKER FULL TIMELINE — {cls._step} checkpoints\n{'█'*76}")
        for e in cls._log:
            print(f"\n  ── Step {e['step']:02d}: {e['label']}")
            for name, val in e["vars"].items():
                print(f"     {name:<68} = {repr(val)}")
        print(f"\n{'█'*76}\n")


# ── ScanEventTemporalMatcher ──────────────────────────────────────────────────

class ScanEventTemporalMatcher:
    SCAN_EVENT_CROSS_PROBE_TEMPORAL_MATCH_TOLERANCE_HOURS = 2

    @staticmethod
    def extract_chronologically_ordered_scan_event_list_from_probe_manifest(manifest):
        events = []; cursor = 1
        while True:
            ev = getattr(manifest, f"scan_event_{cursor}", None)
            if ev is None: break
            events.append({
                "scan_event_sequence_number":                  ev.scan_event_sequence_number,
                "scan_event_physical_location_city_state":     ev.scan_location_city_state,
                "scan_event_timestamp_utc_string":             ev.scan_timestamp_utc,
                "scan_event_timestamp_as_comparable_datetime": datetime.datetime.strptime(ev.scan_timestamp_utc, "%Y-%m-%d %H:%M:%S"),
                "scan_event_id":                               ev.scan_event_id,
            }); cursor += 1
        return sorted(events, key=lambda e: e["scan_event_sequence_number"])

    @staticmethod
    def match_and_group_cross_probe_scan_events_by_time_period_window(primary_list, secondary_list, tolerance_hours):
        tol = datetime.timedelta(hours=tolerance_hours)
        matched = []; consumed = set()
        for pri in primary_list:
            pri_dt = pri["scan_event_timestamp_as_comparable_datetime"]
            best_sec, best_gap, best_idx = None, float("inf"), None
            for idx, sec in enumerate(secondary_list):
                if idx in consumed: continue
                gap = abs(pri_dt - sec["scan_event_timestamp_as_comparable_datetime"])
                if gap <= tol and gap.total_seconds() < best_gap:
                    best_sec = sec; best_gap = gap.total_seconds(); best_idx = idx
            if best_sec is not None:
                consumed.add(best_idx); seq = len(matched) + 1
                matched.append({
                    "matched_group_sequential_number": seq,
                    "temporal_match_tolerance_hours_used_to_form_this_group": tolerance_hours,
                    "matched_group_actual_time_gap_between_probe_events_hours": round(best_gap / 3600, 4),
                    "matched_group_primary_probe_contributing_scan_event": {
                        "scan_event_sequence_number":              pri["scan_event_sequence_number"],
                        "scan_event_physical_location_city_state": pri["scan_event_physical_location_city_state"],
                        "scan_event_timestamp_utc_string":         pri["scan_event_timestamp_utc_string"],
                        "scan_event_id":                           pri["scan_event_id"],
                    },
                    "matched_group_secondary_probe_contributing_scan_event": {
                        "scan_event_sequence_number":              best_sec["scan_event_sequence_number"],
                        "scan_event_physical_location_city_state": best_sec["scan_event_physical_location_city_state"],
                        "scan_event_timestamp_utc_string":         best_sec["scan_event_timestamp_utc_string"],
                        "scan_event_id":                           best_sec["scan_event_id"],
                    },
                })
        return matched, [], [s for i, s in enumerate(secondary_list) if i not in consumed]


# ── JITPipelineArchitecture ───────────────────────────────────────────────────
# FIX: classes assigned inline in the class body so Arch.Data.Models.DynamicProbe,
# Arch.Data.Models.JITObject, Arch.Data.Telemetry.PipelineTelemetry,
# Arch.Observability.StoryLog, and Arch.Observability.VarTracker are all
# resolvable without any post-hoc registration block.

class JITPipelineArchitecture:
    class Core:
        @staticmethod
        def ms_now():
            return datetime.datetime.now().strftime("%H:%M:%S.") + f"{datetime.datetime.now().microsecond//1000:03d}"
    class Data:
        class Models:
            DynamicProbe = DynamicProbe   # ← FIX: assigned at class-body parse time
            JITObject    = JITObject      # ← FIX
        class Telemetry:
            PipelineTelemetry = PipelineTelemetry   # ← FIX
    class Observability:
        StoryLog   = StoryLog    # ← FIX
        VarTracker = VarTracker  # ← FIX
    class Execution:
        class Nodes:       pass
        class Orchestrator: pass


print("Cell 1 — infrastructure loaded.")
print(f"  DynamicSection · DynamicProbe · JITObject")
print(f"  TandemCalc: TC-01..05 · TC-N · TC-K · TC-C · TC-V · TC-P1..4 · TC-PIPE")
print(f"  HF routing table · HFEngine · PipelineTelemetry · StoryLog · VarTracker")
print(f"  ScanEventTemporalMatcher")
print(f"  IsomorphicDistributionEngine: ISO-01..09  stateless  tel-elapsed timer")
print(f"    C = {IsomorphicDistributionEngine.ISO_M}·velocity_normalized + {IsomorphicDistributionEngine.ISO_B}")
print(f"    V_i = C·w_i / N_connected   w_i = 1+sin(2πi/N+π/4)   period = {IsomorphicDistributionEngine.ISO_BASE_PERIOD_SECONDS}s × cumulative_magnitude")
print(f"  JITPipelineArchitecture — Arch.Data.Models.DynamicProbe  ✓")
print(f"                          — Arch.Data.Models.JITObject      ✓")
print(f"                          — Arch.Data.Telemetry.PipelineTelemetry  ✓")
print(f"                          — Arch.Observability.StoryLog     ✓")
print(f"                          — Arch.Observability.VarTracker   ✓")

Cell 1 — infrastructure loaded.
  DynamicSection · DynamicProbe · JITObject
  TandemCalc: TC-01..05 · TC-N · TC-K · TC-C · TC-V · TC-P1..4 · TC-PIPE
  HF routing table · HFEngine · PipelineTelemetry · StoryLog · VarTracker
  ScanEventTemporalMatcher
  IsomorphicDistributionEngine: ISO-01..09  stateless  tel-elapsed timer
    C = 1.5·velocity_normalized + 0.2
    V_i = C·w_i / N_connected   w_i = 1+sin(2πi/N+π/4)   period = 4.0s × cumulative_magnitude
  JITPipelineArchitecture — Arch.Data.Models.DynamicProbe  ✓
                          — Arch.Data.Models.JITObject      ✓
                          — Arch.Data.Telemetry.PipelineTelemetry  ✓
                          — Arch.Observability.StoryLog     ✓
                          — Arch.Observability.VarTracker   ✓


In [2]:
# @title 2: OBSERVABILITY, STORY LOGS, 3D VISUALIZATION & HF TELEMETRY GRAPH

# =============================================================================
# ACT 2: THE INSTRUMENTS — NARRATIVE STORY
# We build every lens we need to see inside the system:
#   • 3D animated telemetry scatter — skew / impact / phase trajectory
#   • Architecture network — probe→JIT→HFEngine routing flow
#   • Hierarchical infrastructure — nested payload tree
#   • HF Telemetry Dashboard — NEW: dedicated visual for every HFEngine.fire()
#     event, showing which rules fired, which TandemCalc functions were called,
#     which dst paths were written, and the THFPI pipeline result values.
# =============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

class Visualizations:

    # ─────────────────────────────────────────────────────────────────────────
    # 1. ANIMATED 3D TELEMETRY SCATTER
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_animated_telemetry(tel, pri, sec):
        phases = list(tel.skews.keys())
        if not phases: print("No telemetry phases to render."); return

        skews   = [tel.skews.get(p, 0) for p in phases]
        impacts = [tel.impacts.get(p, 0) for p in phases]
        durs    = [tel.durs.get(p, 0) / 1e6 for p in phases]  # ns → ms

        frames = []
        for i in range(1, len(phases) + 1):
            frames.append(go.Frame(
                data=[go.Scatter3d(
                    x=skews[:i], y=impacts[:i], z=durs[:i],
                    mode="lines+markers+text",
                    text=phases[:i], textposition="top center",
                    marker=dict(size=6, color=list(range(i)),
                                colorscale="Plasma", showscale=False),
                    line=dict(color="cyan", width=3),
                )],
                name=str(i),
            ))

        fig = go.Figure(
            data=[go.Scatter3d(
                x=skews[:1], y=impacts[:1], z=durs[:1],
                mode="markers+text", text=phases[:1],
                marker=dict(size=8, color="cyan"),
            )],
            frames=frames,
            layout=go.Layout(
                title="⚡ THFPI Pipeline — 3D Phase Trajectory (Skew / Impact / Duration)",
                paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
                font=dict(color="white"),
                scene=dict(
                    xaxis=dict(title="Skew (s)", backgroundcolor="#0d0d2b",
                               gridcolor="#333366", color="white"),
                    yaxis=dict(title="Impact Score", backgroundcolor="#0d0d2b",
                               gridcolor="#333366", color="white"),
                    zaxis=dict(title="Duration (ms)", backgroundcolor="#0d0d2b",
                               gridcolor="#333366", color="white"),
                    bgcolor="#0a0a1a",
                ),
                updatemenus=[dict(
                    type="buttons", showactive=False, y=1.05, x=0.5, xanchor="center",
                    buttons=[
                        dict(label="▶ Play",  method="animate",
                             args=[None, {"frame": {"duration": 600, "redraw": True}, "fromcurrent": True}]),
                        dict(label="⏸ Pause", method="animate",
                             args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
                    ],
                )],
            )
        )
        fig.show()

    # ─────────────────────────────────────────────────────────────────────────
    # 2. ARCHITECTURE NETWORK GRAPH
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_architecture_network(tel, pri, sec):
        nodes = ["PRI Probe", "SEC Probe", "TandemCalc", "HFEngine", "JIT Bus",
                 "Phase_1", "Phase_2", "Phase_3", "Phase_4", "Telemetry"]
        edges = [(0,2),(1,2),(2,3),(3,4),(4,5),(4,6),(4,7),(4,8),(5,9),(6,9),(7,9),(8,9)]

        import math
        n = len(nodes)
        xs = [math.cos(2*math.pi*i/n) for i in range(n)]
        ys = [math.sin(2*math.pi*i/n) for i in range(n)]

        edge_x, edge_y = [], []
        for a, b in edges:
            edge_x += [xs[a], xs[b], None]; edge_y += [ys[a], ys[b], None]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                                  line=dict(color="#334499", width=1.5), hoverinfo="none"))
        colors = ["#00d2ff","#00d2ff","#ff6b6b","#ffd700","#7fff7f",
                  "#c084fc","#c084fc","#c084fc","#c084fc","#f97316"]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="markers+text",
                                  text=nodes, textposition="top center",
                                  marker=dict(size=20, color=colors,
                                              line=dict(color="white", width=1)),
                                  hoverinfo="text"))
        fig.update_layout(
            title="🔗 THFPI Architecture — Component Routing Network",
            showlegend=False, paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
            font=dict(color="white"),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=500,
        )
        fig.show()

    # ─────────────────────────────────────────────────────────────────────────
    # 3. HIERARCHICAL INFRASTRUCTURE SUNBURST
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_hierarchical_infrastructure(jit, pri, sec):
        labels, parents, values, colors = ["THFPI"], [""], [1], ["#0a0a1a"]
        palette = {"PRI":"#00d2ff","SEC":"#ff6b6b","JIT":"#7fff7f","TI":"#ffd700","HF":"#c084fc"}

        def walk(obj, parent, color, depth=0):
            if depth > 3: return
            data = obj.to_dict() if isinstance(obj, DynamicSection) else \
                   {k: v for k, v in obj.__dict__.items() if not k.startswith("_")} if hasattr(obj, "__dict__") else {}
            for k, v in list(data.items())[:12]:
                label = f"{k[:20]}"
                if label in labels: label = f"{label}_{depth}"
                labels.append(label); parents.append(parent)
                values.append(1); colors.append(color)
                if isinstance(v, (DynamicSection,)) or (hasattr(v, '__dict__') and not isinstance(v, type)):
                    walk(v, label, color, depth+1)

        walk(pri.payload, "THFPI", palette["PRI"])
        walk(sec.payload, "THFPI", palette["SEC"])
        walk(jit.jit_metrics, "THFPI", palette["JIT"])
        walk(jit.tandem_interface, "THFPI", palette["TI"])

        fig = go.Figure(go.Sunburst(
            labels=labels, parents=parents, values=values,
            marker=dict(colors=colors, line=dict(color="black", width=0.5)),
            branchvalues="total",
        ))
        fig.update_layout(
            title="🌐 THFPI Hierarchical Infrastructure — Payload & Interface Tree",
            paper_bgcolor="#0a0a1a", font=dict(color="white"), height=600,
        )
        fig.show()

    # ─────────────────────────────────────────────────────────────────────────
    # 4. HF TELEMETRY DASHBOARD  ← NEW
    # Visualises every HFEngine.fire() event as a dedicated subplot grid:
    #   Top row    — Bar chart: rules fired per event (height = n_rules)
    #   Middle row — Timeline: fire events on a horizontal T-axis
    #   Bottom row — THFPI pipeline centroid geometry (Group 1 centroids)
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_hf_telemetry_dashboard(tel, jit):
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                "HF Fire Events — Rules Activated per Call",
                "TandemCalc Functions Called per Event",
                "HF Event Timeline (T-offset from pipeline start)",
                "HF Destination Paths Written per Event",
                "THFPI Group 1 — Centroid Geometry",
                "THFPI Group 1 — Conjunctive Delta vs Raw Gap",
            ),
            vertical_spacing=0.14, horizontal_spacing=0.10,
        )

        events   = tel.hf_event_log
        if not events:
            print("No HF events recorded — pipeline may not have fired yet."); return

        labels   = [f"{e['node']}\n{e['moment']}" for e in events]
        n_rules  = [e["n_rules"] for e in events]
        t_offsets_ms = [(e["ts_ns"] - tel.start_ns) / 1e6 for e in events]
        fn_labels = [", ".join(set(e["fn_names"])) for e in events]
        dst_labels = ["\n".join(e["dst_paths"])[:40] for e in events]

        # ── Row 1 Left: rules per event bar ──
        bar_colors = ["#00d2ff" if n == 1 else "#ffd700" if n == 2 else "#ff6b6b" for n in n_rules]
        fig.add_trace(go.Bar(
            x=labels, y=n_rules, marker_color=bar_colors,
            text=n_rules, textposition="outside",
            name="Rules fired", showlegend=False,
        ), row=1, col=1)
        fig.update_yaxes(title_text="# Rules", row=1, col=1)

        # ── Row 1 Right: fn names per event ──
        fig.add_trace(go.Bar(
            x=labels, y=[1]*len(events), text=fn_labels,
            textposition="inside", marker_color="#c084fc",
            name="Fn called", showlegend=False,
        ), row=1, col=2)

        # ── Row 2 Left: timeline scatter ──
        fig.add_trace(go.Scatter(
            x=t_offsets_ms, y=[0]*len(events), mode="markers+text",
            text=labels, textposition="top center",
            marker=dict(size=14, color=bar_colors,
                        line=dict(color="white", width=1)),
            name="Timeline", showlegend=False,
        ), row=2, col=1)
        fig.update_xaxes(title_text="T-offset (ms)", row=2, col=1)
        fig.update_yaxes(visible=False, row=2, col=1)

        # ── Row 2 Right: dst paths per event ──
        fig.add_trace(go.Bar(
            x=labels, y=[1]*len(events), text=dst_labels,
            textposition="inside", marker_color="#7fff7f",
            name="Dst paths", showlegend=False,
        ), row=2, col=2)

        # ── Row 3: THFPI Group 1 centroid geometry ──
        _g = jit.tandem_interface.thfpi_group1 if jit and jit.tandem_interface.thfpi_group1 else {}
        if _g:
            c0   = _g.get("kmeans_centroid_0_primary",            0)
            c1   = _g.get("kmeans_centroid_1_synthetic_midpoint", 0)
            c2   = _g.get("kmeans_centroid_2_secondary",          0)
            conj = _g.get("delta_conjunctive_median",             0)
            raw  = abs(_g.get("norm_signed_delta", 0))

            # centroid number line
            fig.add_trace(go.Scatter(
                x=[c0, c1, c2],
                y=[0, 0, 0],
                mode="markers+text",
                text=[f"C0 PRIMARY<br>{c0:.6f}", f"C1 SYNTHETIC<br>{c1:.6f}", f"C2 SECONDARY<br>{c2:.6f}"],
                textposition=["bottom center","top center","bottom center"],
                marker=dict(size=[14, 18, 14],
                            color=["#00d2ff", "#ffd700", "#ff6b6b"],
                            symbol=["circle","star","circle"],
                            line=dict(color="white", width=1)),
                name="Centroids", showlegend=False,
            ), row=3, col=1)
            # gap annotation lines
            fig.add_shape(type="line", x0=c0, x1=c1, y0=0, y1=0,
                          line=dict(color="#ffd700", width=3, dash="dot"),
                          row=3, col=1, xref="x5", yref="y5")
            fig.add_shape(type="line", x0=c1, x1=c2, y0=0, y1=0,
                          line=dict(color="#ffd700", width=3, dash="dot"),
                          row=3, col=1, xref="x5", yref="y5")
            fig.update_xaxes(title_text="Normalized day-fraction", row=3, col=1)
            fig.update_yaxes(visible=False, row=3, col=1)

            # conjunctive vs raw gap bar comparison
            fig.add_trace(go.Bar(
                x=["Raw inter-probe gap", "Conjunctive median delta\n(half-gap, NOVEL)"],
                y=[raw * 1440, conj * 1440],
                marker_color=["#ff6b6b", "#ffd700"],
                text=[f"{raw*1440:.4f} min", f"{conj*1440:.4f} min"],
                textposition="outside",
                name="Delta comparison", showlegend=False,
            ), row=3, col=2)
            fig.update_yaxes(title_text="Minutes", row=3, col=2)
        else:
            fig.add_annotation(text="THFPI Group 1 not yet computed",
                               xref="paper", yref="paper", x=0.25, y=0.1,
                               showarrow=False, font=dict(color="white"))

        fig.update_layout(
            title="📡 THFPI HF Telemetry Dashboard — Every fire() Event Visualised",
            paper_bgcolor="#0a0a1a", plot_bgcolor="#0d0d2b",
            font=dict(color="white"),
            height=900,
        )
        fig.update_xaxes(color="white", gridcolor="#333366")
        fig.update_yaxes(color="white", gridcolor="#333366")
        fig.show()

JITPipelineArchitecture.Observability.Visualizations = Visualizations

In [3]:
# @title 3: FACTORY PHASE — 1 (Ingestion)
# =============================================================================
# ACT 3: THE JOURNEY BEGINS (Phase 1) — NARRATIVE STORY
# Arbitrary objects enter the dynamic factory. JIT receives its first HF values.
# We calculate initial physical skews and prove the JIT embedded functions work.
# VarTracker snapshots capture every key moment inside the factory.
# =============================================================================

class Factory_Phase_1:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log = JITPipelineArchitecture.Observability.StoryLog
        VT  = JITPipelineArchitecture.Observability.VarTracker

        # [1] tel enters Phase_1 — starts the phase clock.
        uid = tel.enter("Phase_1")
        setattr(jit, 'phase_1', DynamicSection())
        setattr(pri, 'phase_1', DynamicSection())
        setattr(sec, 'phase_1', DynamicSection())

        # ── BREAKPOINT 1 ── Phase entry, probes and JIT constructed ──────────
        VT.snapshot("PHASE 1 ENTRY — jit, pri, sec constructed; tel clock started", {
            "phase_1_entry_uid":            uid,
            "jit_uid":                      jit.uid,
            "pri_uid":                      pri.uid,
            "sec_uid":                      sec.uid,
            "pri_payload_calc_value":       pri.payload.calc_value,
            "sec_payload_calc_value":       sec.payload.calc_value,
            "pri_payload_step1_id":         pri.payload.step1_id,
            "pri_payload_step1_time":       pri.payload.step1_time,
            "sec_payload_step1_id":         sec.payload.step1_id,
            "sec_payload_step1_time":       sec.payload.step1_time,
            "jit_tandem_interface_results": jit.tandem_interface.results,
        })

        # ◀◀ THFPI ACTIVATION POINT 1/3 — on_entry fires here
        HFEngine.fire("Phase_1", "on_entry", jit, pri, sec, tel)
        # [TEL-HF2] Record this fire() event in the structured telemetry log.
        tel.record_hf_event("Phase_1", "on_entry", HFEngine.last_retrieval["reads"])

        # ── BREAKPOINT 2 ── HF on_entry tandem retrieval + full payload ──────
        _ledger = HFEngine.last_retrieval
        _r0     = _ledger["reads"][0] if _ledger["reads"] else {}
        VT.snapshot("HF on_entry — TANDEM RETRIEVAL + FULL PROBE PAYLOAD", {
            "hf_on_entry_node":                             _ledger.get("node"),
            "hf_on_entry_moment":                           _ledger.get("moment"),
            "hf_on_entry_src_pri_path":                     _r0.get("src_pri_path"),
            "hf_on_entry_vp_retrieved":                     _r0.get("vp_retrieved"),
            "hf_on_entry_src_sec_path":                     _r0.get("src_sec_path"),
            "hf_on_entry_vs_retrieved":                     _r0.get("vs_retrieved"),
            "hf_on_entry_fn_called":                        _r0.get("fn_called"),
            "hf_on_entry_result_on_tandem_interface":       _r0.get("result"),
            "hf_on_entry_dst_path_written":                 _r0.get("dst_path"),
            "hf_on_entry_expression":                       _r0.get("expression"),
            "── PRI PROBE ACCESSIBLE DATA ──":              "─" * 20,
            "pri_probe_calc_value":                         pri.payload.calc_value,
            "pri_probe_step1_id":                           pri.payload.step1_id,
            "pri_probe_step1_time":                         pri.payload.step1_time,
            "pri_probe_step2_id":                           pri.payload.step2_id,
            "pri_probe_step3_id":                           pri.payload.step3_id,
            "pri_probe_step4_id":                           pri.payload.step4_id,
            "pri_probe_manifest_tracking_id":               pri.payload.package_shipping_manifest.package_tracking_id,
            "pri_probe_manifest_scan_event_1":              pri.payload.package_shipping_manifest.scan_event_1,
            "pri_probe_manifest_scan_event_2":              pri.payload.package_shipping_manifest.scan_event_2,
            "pri_probe_manifest_scan_event_3":              pri.payload.package_shipping_manifest.scan_event_3,
            "pri_probe_hf_on_entry_expr_stamped":           pri.payload.hf_expr_Phase_1_on_entry,
            "── SEC PROBE ACCESSIBLE DATA ──":              "─" * 20,
            "sec_probe_calc_value":                         sec.payload.calc_value,
            "sec_probe_step1_id":                           sec.payload.step1_id,
            "sec_probe_step1_time":                         sec.payload.step1_time,
            "sec_probe_manifest_tracking_id":               sec.payload.package_shipping_manifest.package_tracking_id,
            "sec_probe_manifest_scan_event_1":              sec.payload.package_shipping_manifest.scan_event_1,
            "sec_probe_hf_on_entry_expr_stamped":           sec.payload.hf_expr_Phase_1_on_entry,
            "── JIT STATE ──":                              "─" * 20,
            "jit_tandem_interface_results_after_on_entry":  jit.tandem_interface.results,
            "tel_hf_log":                                   tel.hf_log,
        })

        # [4] stamp phase events
        pri.phase_1.event = "Ingestion_A_Mapped"
        sec.phase_1.event = "Ingestion_B_Mapped"
        jit.phase_1.entry_uid = uid

        # [5] lift calc values
        primary_probe_phase_calculation_value   = getattr(pri.payload, 'calc_value', 0)
        secondary_probe_phase_calculation_value = getattr(sec.payload, 'calc_value', 0)
        primary_probe_belt_step_1_scan_id       = getattr(pri.payload, 'step1_id', 'A0X')

        # ── BREAKPOINT 3 ── Probe values retrieved ────────────────────────────
        VT.snapshot("PROBE VALUES RETRIEVED FROM BOTH PROBE PAYLOADS", {
            "primary_probe_phase_calculation_value":        primary_probe_phase_calculation_value,
            "secondary_probe_phase_calculation_value":      secondary_probe_phase_calculation_value,
            "primary_probe_belt_step_1_scan_id":            primary_probe_belt_step_1_scan_id,
            "pri_probe_phase_1_event_label":                pri.phase_1.event,
            "sec_probe_phase_1_event_label":                sec.phase_1.event,
            "jit_phase_1_entry_uid":                        jit.phase_1.entry_uid,
        })

        # [7] JIT embedded regex
        phase_1_embedded_regex_numeric_extraction_result = jit.evaluate_regex(r"\d+", primary_probe_belt_step_1_scan_id)

        # ── BREAKPOINT 4 ── After JIT regex ───────────────────────────────────
        VT.delta("AFTER JIT evaluate_regex — numeric extraction from belt step 1 scan id", {
            "phase_1_embedded_regex_numeric_extraction_result": phase_1_embedded_regex_numeric_extraction_result,
            "primary_probe_belt_step_1_scan_id_input":          primary_probe_belt_step_1_scan_id,
            "jit_metrics_embedded_results":                     jit.jit_metrics.embedded_results,
        })

        # [8] manual TandemCalc product
        phase_1_manual_tandem_cross_probe_product_result = TandemCalc.manual_phase_calc(
            primary_probe_phase_calculation_value, secondary_probe_phase_calculation_value)

        # ── BREAKPOINT 5 ── After manual tandem calc ──────────────────────────
        VT.delta("AFTER TandemCalc.manual_phase_calc — cross-probe product computed", {
            "phase_1_manual_tandem_cross_probe_product_result": phase_1_manual_tandem_cross_probe_product_result,
            "primary_probe_phase_calculation_value":            primary_probe_phase_calculation_value,
            "secondary_probe_phase_calculation_value":          secondary_probe_phase_calculation_value,
            "formula_string": f"{primary_probe_phase_calculation_value} * {secondary_probe_phase_calculation_value} = {phase_1_manual_tandem_cross_probe_product_result}",
        })

        # [9] store into jit
        jit.jit_metrics.embedded_results.phase_1_embedded_regex_numeric_extraction_result = phase_1_embedded_regex_numeric_extraction_result
        jit.tandem_interface.results.phase_1_manual_tandem_cross_probe_product_result      = phase_1_manual_tandem_cross_probe_product_result

        # ── BREAKPOINT 6 ── After JIT stores results ──────────────────────────
        VT.delta("AFTER JIT stores regex and tandem product results into jit storage", {
            "jit_metrics_embedded_results":             jit.jit_metrics.embedded_results,
            "jit_tandem_interface_results":             jit.tandem_interface.results,
        })

        # [10-15] skew + impact
        primary_probe_belt_step1_wall_clock_offset_seconds   = getattr(pri.payload, 'step1_time', 0)
        secondary_probe_belt_step1_wall_clock_offset_seconds = getattr(sec.payload, 'step1_time', 0)
        cross_probe_belt_step1_absolute_wall_clock_duration_seconds = abs(
            primary_probe_belt_step1_wall_clock_offset_seconds - secondary_probe_belt_step1_wall_clock_offset_seconds)
        combined_probe_phase_calculation_value_mass = (
            getattr(pri.payload, 'calc_value', 1) + getattr(sec.payload, 'calc_value', 1))
        tandem_product_carried_forward_as_impact_scale_factor = phase_1_manual_tandem_cross_probe_product_result
        signed_cross_probe_belt_step1_arrival_time_skew_seconds = TandemCalc.subtract(
            primary_probe_belt_step1_wall_clock_offset_seconds,
            secondary_probe_belt_step1_wall_clock_offset_seconds)
        phase_1_impact_score = (
            (cross_probe_belt_step1_absolute_wall_clock_duration_seconds * combined_probe_phase_calculation_value_mass)
            + (abs(signed_cross_probe_belt_step1_arrival_time_skew_seconds) * tandem_product_carried_forward_as_impact_scale_factor))

        # ── BREAKPOINT 7 ── All metrics computed ──────────────────────────────
        VT.snapshot("ALL PHASE 1 METRICS COMPUTED — ready to commit to telemetry", {
            "primary_probe_belt_step1_wall_clock_offset_seconds":            primary_probe_belt_step1_wall_clock_offset_seconds,
            "secondary_probe_belt_step1_wall_clock_offset_seconds":          secondary_probe_belt_step1_wall_clock_offset_seconds,
            "cross_probe_belt_step1_absolute_wall_clock_duration_seconds":   cross_probe_belt_step1_absolute_wall_clock_duration_seconds,
            "combined_probe_phase_calculation_value_mass":                   combined_probe_phase_calculation_value_mass,
            "tandem_product_carried_forward_as_impact_scale_factor":         tandem_product_carried_forward_as_impact_scale_factor,
            "signed_cross_probe_belt_step1_arrival_time_skew_seconds":       signed_cross_probe_belt_step1_arrival_time_skew_seconds,
            "phase_1_impact_score":                                          phase_1_impact_score,
            "phase_1_impact_score_formula_string": (
                f"({cross_probe_belt_step1_absolute_wall_clock_duration_seconds}"
                f"*{combined_probe_phase_calculation_value_mass})"
                f" + ({abs(signed_cross_probe_belt_step1_arrival_time_skew_seconds)}"
                f"*{tandem_product_carried_forward_as_impact_scale_factor})"
                f" = {phase_1_impact_score}"
            ),
        })

        # [16] commit to telemetry
        tel.record_metrics("Phase_1", signed_cross_probe_belt_step1_arrival_time_skew_seconds, phase_1_impact_score)

        # [17] story log
        Log.trace("Phase_1",
            "Acquisition of values from PRI and SEC. Execution of embedded JIT regex. Manual execution of separate Tandem function complete.",
            [f"JIT Embedded Regex r'\\d+' on '{primary_probe_belt_step_1_scan_id}' -> {phase_1_embedded_regex_numeric_extraction_result}",
             f"Tandem Manual phase_calc({primary_probe_phase_calculation_value}, {secondary_probe_phase_calculation_value}) -> {phase_1_manual_tandem_cross_probe_product_result}"],
            {"Skew": signed_cross_probe_belt_step1_arrival_time_skew_seconds,
             "Manual_Tandem": phase_1_manual_tandem_cross_probe_product_result,
             "Impact": phase_1_impact_score},
            jit, pri, sec, tel)

        # ◀◀ THFPI ACTIVATION POINT 2/3 — on_exit fires here
        HFEngine.fire("Phase_1", "on_exit", jit, pri, sec, tel)
        tel.record_hf_event("Phase_1", "on_exit", HFEngine.last_retrieval["reads"])

        # ── BREAKPOINT 8 ── HF on_exit tandem retrieval + full payload ────────
        _ledger_exit = HFEngine.last_retrieval
        _r0e         = _ledger_exit["reads"][0] if _ledger_exit["reads"] else {}
        VT.snapshot("HF on_exit — TANDEM RETRIEVAL + FULL PROBE + JIT STATE", {
            "hf_on_exit_node":                              _ledger_exit.get("node"),
            "hf_on_exit_moment":                            _ledger_exit.get("moment"),
            "hf_on_exit_src_pri_path":                      _r0e.get("src_pri_path"),
            "hf_on_exit_vp_retrieved":                      _r0e.get("vp_retrieved"),
            "hf_on_exit_src_sec_path":                      _r0e.get("src_sec_path"),
            "hf_on_exit_vs_retrieved":                      _r0e.get("vs_retrieved"),
            "hf_on_exit_fn_called":                         _r0e.get("fn_called"),
            "hf_on_exit_result_on_tandem_interface":        _r0e.get("result"),
            "hf_on_exit_dst_path_written":                  _r0e.get("dst_path"),
            "hf_on_exit_expression":                        _r0e.get("expression"),
            "── PRI PROBE ──":                              "─" * 20,
            "pri_probe_hf_on_exit_expr_stamped":            pri.payload.hf_expr_Phase_1_on_exit,
            "── SEC PROBE ──":                              "─" * 20,
            "sec_probe_hf_on_exit_expr_stamped":            sec.payload.hf_expr_Phase_1_on_exit,
            "── JIT STATE ──":                              "─" * 20,
            "jit_tandem_interface_results_after_on_exit":   jit.tandem_interface.results,
            "tel_skews_dict":                               tel.skews,
            "tel_impacts_dict":                             tel.impacts,
            "tel_hf_log":                                   tel.hf_log,
        })

        # [19] stop phase clock
        tel.exit("Phase_1")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_1 = Factory_Phase_1

In [4]:
# @title 4: FACTORY PHASE — 2 (Transformation)
# =============================================================================
# ACT 4: DIVERGENCE (Phase 2) - NARRATIVE STORY
# Packages transition through transformation layers. We use tandem subtractions
# to measure their distance apart and store it universally dynamically.
# =============================================================================

class Factory_Phase_2:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log, Arch = JITPipelineArchitecture.Observability.StoryLog, JITPipelineArchitecture
        uid = tel.enter("Phase_2")

        setattr(jit, 'phase_2', DynamicSection())
        setattr(pri, 'phase_2', DynamicSection())
        setattr(sec, 'phase_2', DynamicSection())

        # HF Matrix automatically fires the checkpoint_entry Tandem function!
        HFEngine.fire("Phase_2", "on_entry", jit, pri, sec, tel)

        pri.phase_2.event = "Transform_A_Logged"
        sec.phase_2.event = "Transform_B_Logged"
        jit.phase_2.entry_uid = uid

        # 115: Retrieve the next set of values from the probes.
        val_pri = getattr(pri.payload, 'calc_value', 0)
        val_sec = getattr(sec.payload, 'calc_value', 0)
        shipping_id_sec = getattr(sec.payload, 'step2_id', 'B0Y')

        # 116: Select the JIT object and invoke the embedded linear regex function.
        regex_result = jit.evaluate_regex(r"\d+", shipping_id_sec)

        # 117: Call the manual Tandem function explicitly inside the factory stage.
        manual_tandem_result = TandemCalc.manual_phase_calc(val_pri, val_sec)

        # 118: Store the manual results into the JIT architecture.
        jit.jit_metrics.embedded_results.phase_2_regex = regex_result
        jit.tandem_interface.results.phase_2_manual_calc = manual_tandem_result

        off_p = getattr(pri.payload, 'step2_time', 0)
        off_s = getattr(sec.payload, 'step2_time', 0)
        dur_p = off_p - getattr(pri.payload, 'step1_time', 0)
        dur_s = off_s - getattr(sec.payload, 'step1_time', 0)
        wall_duration = abs(dur_p + dur_s)
        obj_mass = getattr(pri.payload, 'calc_value', 1)
        tandem_conn = abs(manual_tandem_result) if manual_tandem_result != 0 else 1
        skew = TandemCalc.subtract(off_p, off_s)
        impact = (wall_duration * obj_mass) / tandem_conn
        tel.record_metrics("Phase_2", skew, impact)

        narr = f"Objects route through active divergence sectors (Running in PARALLEL with Phase 3). Execution of JIT Embedded Regex and Tandem Interface manual phase function complete."
        ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]
        Log.trace("Phase_2", narr, ops, {"Route_Drift": skew, "Impact": impact}, jit, pri, sec, tel)

        # HF Matrix automatically fires the checkpoint_exit Tandem function!
        HFEngine.fire("Phase_2", "on_exit", jit, pri, sec, tel)

        tel.exit("Phase_2")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_2 = Factory_Phase_2

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_511/3320084938.py:52: SyntaxWarning: invalid escape sequence '\d'
  ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]


In [5]:
# @title 5: FACTORY PHASE — 3 (Synthesis)
# =============================================================================
# ACT 5: SYNTHESIS (Phase 3) - NARRATIVE STORY
# The objects logically meet back up. We dynamically calculate total accumulated
# mass (addition) and prove cross-verification using dynamic property lookups.
# =============================================================================

class Factory_Phase_3:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log, Arch = JITPipelineArchitecture.Observability.StoryLog, JITPipelineArchitecture
        uid = tel.enter("Phase_3")

        setattr(jit, 'phase_3', DynamicSection())
        setattr(pri, 'phase_3', DynamicSection())
        setattr(sec, 'phase_3', DynamicSection())

        # HF Matrix automatically fires the checkpoint_entry Tandem function!
        HFEngine.fire("Phase_3", "on_entry", jit, pri, sec, tel)

        pri.phase_3.event = "Sync_A_Aligned"
        sec.phase_3.event = "Sync_B_Aligned"
        jit.phase_3.entry_uid = uid

        # 119: Retrieve values from the probe payloads.
        val_pri = getattr(pri.payload, 'calc_value', 0)
        val_sec = getattr(sec.payload, 'calc_value', 0)
        shipping_time_pri = getattr(pri.payload, 'step3_time', 0.0)

        # 120: Select the JIT object and invoke the embedded linear regex function.
        regex_result = jit.evaluate_regex(r"\d+", shipping_time_pri)

        # 121: Call the manual Tandem function explicitly.
        manual_tandem_result = TandemCalc.manual_phase_calc(val_pri, val_sec)

        # 122: Store results into the JIT architecture.
        jit.jit_metrics.embedded_results.phase_3_regex = regex_result
        jit.tandem_interface.results.phase_3_manual_calc = manual_tandem_result

        off_p = getattr(pri.payload, 'step3_time', 0)
        off_s = getattr(sec.payload, 'step3_time', 0)
        dur_p = off_p - getattr(pri.payload, 'step2_time', 0)
        dur_s = off_s - getattr(sec.payload, 'step2_time', 0)
        wall_duration = abs(dur_p + dur_s)
        obj_mass = getattr(pri.payload, 'calc_value', 1)
        tandem_conn = manual_tandem_result
        skew = TandemCalc.subtract(off_p, off_s)
        impact = (wall_duration * obj_mass) + (tandem_conn * abs(skew))
        tel.record_metrics("Phase_3", skew, impact)

        narr = "Probes synchronize states (Running in PARALLEL with Phase 2). Linear Regex captures shipping timestamps. Manual execution of tandem formula explicitly recorded."
        ops =[f"JIT Embedded Regex r'\d+' on '{shipping_time_pri}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]
        Log.trace("Phase_3", narr, ops, {"Sync_Window": skew, "Manual_Tandem_Result": manual_tandem_result, "Impact": impact}, jit, pri, sec, tel)

        # HF Matrix automatically fires the checkpoint_exit Tandem function!
        HFEngine.fire("Phase_3", "on_exit", jit, pri, sec, tel)

        tel.exit("Phase_3")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_3 = Factory_Phase_3

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_511/1896917041.py:52: SyntaxWarning: invalid escape sequence '\d'
  ops =[f"JIT Embedded Regex r'\d+' on '{shipping_time_pri}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]


In [6]:
# @title 6: FACTORY PHASE — 4 (Terminus)
# =============================================================================
# ACT 6: TERMINUS (Phase 4) - NARRATIVE STORY
# The factory journey ends. Final tandem multiplications are run dynamically,
# the objects are sealed, and the JIT memory object is preserved for parsing.
# =============================================================================

class Factory_Phase_4:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log, Arch = JITPipelineArchitecture.Observability.StoryLog, JITPipelineArchitecture
        uid = tel.enter("Phase_4")

        setattr(jit, 'phase_4', DynamicSection())
        setattr(pri, 'phase_4', DynamicSection())
        setattr(sec, 'phase_4', DynamicSection())

        # HF Matrix automatically fires the checkpoint_entry Tandem function!
        HFEngine.fire("Phase_4", "on_entry", jit, pri, sec, tel)

        pri.phase_4.event = "Outbound_A_Secured"
        sec.phase_4.event = "Outbound_B_Secured"
        jit.phase_4.entry_uid = uid

        # 123: Retrieve final values from the probes.
        val_pri = getattr(pri.payload, 'calc_value', 0)
        val_sec = getattr(sec.payload, 'calc_value', 0)
        shipping_id_sec = getattr(sec.payload, 'step4_id', 'B0Y')

        # 124: Select the JIT object and invoke the embedded linear regex function.
        regex_result = jit.evaluate_regex(r"\d+", shipping_id_sec)

        # 125: Call the manual Tandem function explicitly to seal the phase.
        manual_tandem_result = TandemCalc.manual_phase_calc(val_pri, val_sec)

        # 126: Store the final manual results into the JIT architecture.
        jit.jit_metrics.embedded_results.phase_4_regex = regex_result
        jit.tandem_interface.results.phase_4_manual_calc = manual_tandem_result

        off_p = getattr(pri.payload, 'step4_time', 0)
        off_s = getattr(sec.payload, 'step4_time', 0)
        dur_p = off_p - getattr(pri.payload, 'step3_time', 0)
        dur_s = off_s - getattr(sec.payload, 'step3_time', 0)
        wall_duration = abs(dur_p + dur_s)
        obj_mass = getattr(pri.payload, 'calc_value', 1)
        tandem_conn = manual_tandem_result
        skew = TandemCalc.subtract(off_p, off_s)
        impact = (wall_duration * obj_mass) + tandem_conn - abs(skew)
        tel.record_metrics("Phase_4", skew, impact)

        narr = "Final tandem lock achieved. Extraction of final shipping IDs complete. Execution of separate manual tandem function securely tracked."
        ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]
        Log.trace("Phase_4", narr, ops, {"Final_Lock": manual_tandem_result, "Impact": impact}, jit, pri, sec, tel)

        # HF Matrix automatically fires the checkpoint_exit Tandem function!
        HFEngine.fire("Phase_4", "on_exit", jit, pri, sec, tel)

        Log.trace("Phase_4", f"Terminus Lock[{manual_tandem_result}]. Probes Released safely past bounds.",[], {}, jit, pri, sec, tel)

        tel.exit("Phase_4")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_4 = Factory_Phase_4

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_511/965404433.py:52: SyntaxWarning: invalid escape sequence '\d'
  ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]


In [7]:
# @title 7: ORCHESTRATOR — [ORC-M] manifest conversion · [ORC-T] temporal matching · phase sequencing · ACTIVATION POINT 3/3

import concurrent.futures
import time

# ─────────────────────────────────────────────────────────────────────────────
# LOG HELPERS — module-level pure functions, zero side-effects on pipeline state
# ─────────────────────────────────────────────────────────────────────────────

orchestrator_log_box_width_characters          = 84
orchestrator_log_horizontal_rule_string        = "─" * orchestrator_log_box_width_characters

def _print_section_header_box(section_title_string):
    right_pad_space_count = orchestrator_log_box_width_characters - 4 - len(section_title_string)
    print(f"\n╔{'═'*(orchestrator_log_box_width_characters-2)}╗")
    print(f"║  {section_title_string}{' '*max(0,right_pad_space_count)}  ║")
    print(f"╚{'═'*(orchestrator_log_box_width_characters-2)}╝")

def _print_subsection_header_box(subsection_title_string):
    right_pad_space_count = orchestrator_log_box_width_characters - 6 - len(subsection_title_string)
    print(f"  ┌{'─'*(orchestrator_log_box_width_characters-4)}┐")
    print(f"  │  {subsection_title_string}{' '*max(0,right_pad_space_count)}  │")
    print(f"  └{'─'*(orchestrator_log_box_width_characters-4)}┘")

def _print_labeled_value_row(field_label_string, field_value_string, left_indent_spaces=4):
    padded_label = f"{' '*left_indent_spaces}{field_label_string:<44}"
    print(f"{padded_label}  {field_value_string}")

def _compute_elapsed_milliseconds_since(wall_clock_reference_ns):
    return (time.perf_counter_ns() - wall_clock_reference_ns) / 1_000_000.0

def _log_wall_clock_timing_boundary(
        boundary_label_string,
        pipeline_wall_clock_start_ns,
        phase_wall_clock_start_ns=None):
    absolute_elapsed_ms = _compute_elapsed_milliseconds_since(pipeline_wall_clock_start_ns)
    phase_elapsed_ms    = (_compute_elapsed_milliseconds_since(phase_wall_clock_start_ns)
                           if phase_wall_clock_start_ns is not None else None)
    timing_line = f"  ⏱  {boundary_label_string:<52}  T+{absolute_elapsed_ms:9.3f}ms"
    if phase_elapsed_ms is not None:
        timing_line += f"  (phase Δ {phase_elapsed_ms:.3f}ms)"
    print(timing_line)

def _log_hfengine_fire_event(
        fire_event_tag_string,
        hf_node_name_string,
        hf_moment_name_string,
        hfengine_last_retrieval_reads_list,
        pipeline_wall_clock_start_ns):
    print(f"\n  {'▶'*3} ENGINE FIRE [{fire_event_tag_string}]  "
          f"node={hf_node_name_string}  moment={hf_moment_name_string}  "
          f"T+{_compute_elapsed_milliseconds_since(pipeline_wall_clock_start_ns):.3f}ms  "
          f"rules={len(hfengine_last_retrieval_reads_list)}")
    for rule_index, single_rule_read_dict in enumerate(hfengine_last_retrieval_reads_list):
        print(f"  {'─'*(orchestrator_log_box_width_characters-2)}")
        print(f"    rule [{rule_index}]  "
              f"fn={single_rule_read_dict.get('fn_called','?')}  "
              f"note={single_rule_read_dict.get('rule_note','')}")
        _print_labeled_value_row("src_pri_path",  single_rule_read_dict.get("src_pri_path",  "?"))
        _print_labeled_value_row("vp_retrieved",  repr(single_rule_read_dict.get("vp_retrieved", "?")))
        _print_labeled_value_row("src_sec_path",  single_rule_read_dict.get("src_sec_path",  "?"))
        _print_labeled_value_row("vs_retrieved",  repr(single_rule_read_dict.get("vs_retrieved", "?")))
        _print_labeled_value_row("dst_path",      single_rule_read_dict.get("dst_path",      "?"))
        rule_result_value = single_rule_read_dict.get("result", "?")
        if isinstance(rule_result_value, dict):
            print(f"    {'result':<44}  <dict — {len(rule_result_value)} keys>")
            for result_key, result_val in list(rule_result_value.items())[:8]:
                _print_labeled_value_row(
                    f"  .{result_key}",
                    f"{result_val:.10f}" if isinstance(result_val, float) else repr(result_val))
            if len(rule_result_value) > 8:
                print(f"    {'  ...':<44}  ({len(rule_result_value)-8} more keys)")
        else:
            _print_labeled_value_row("result", repr(rule_result_value))
        _print_labeled_value_row("expression", single_rule_read_dict.get("expression", "?"))
    print(f"  ◀◀◀ END FIRE [{fire_event_tag_string}]")

def _log_augmentation_normalization_block(thfpi_group1_composable_pipeline_result_dict):
    _print_subsection_header_box("AUGMENTATION  TC-N  Normalize to Day Fraction")
    _print_labeled_value_row(
        "norm_primary  (pri epoch - midnight) / 86400",
        f"{thfpi_group1_composable_pipeline_result_dict.get('norm_primary', 0):.10f}  "
        f"[{thfpi_group1_composable_pipeline_result_dict.get('norm_primary', 0)*1440:.4f} min from midnight]")
    _print_labeled_value_row(
        "norm_secondary",
        f"{thfpi_group1_composable_pipeline_result_dict.get('norm_secondary', 0):.10f}  "
        f"[{thfpi_group1_composable_pipeline_result_dict.get('norm_secondary', 0)*1440:.4f} min from midnight]")
    _print_labeled_value_row(
        "norm_signed_delta  (pri - sec)",
        f"{thfpi_group1_composable_pipeline_result_dict.get('norm_signed_delta', 0):.10f}  "
        f"[{thfpi_group1_composable_pipeline_result_dict.get('norm_signed_delta', 0)*1440:.4f} min]")
    _print_labeled_value_row(
        "norm_anchor_utc",
        str(thfpi_group1_composable_pipeline_result_dict.get("norm_anchor_utc_string", "")))
    _print_labeled_value_row(
        "norm_anchor_epoch",
        str(thfpi_group1_composable_pipeline_result_dict.get("norm_anchor_epoch", "")))

def _log_augmentation_kmeans_block(thfpi_group1_composable_pipeline_result_dict):
    _print_subsection_header_box("AUGMENTATION  TC-K  k-Means Three-Centroid Synthetic")
    _print_labeled_value_row(
        "synthetic_seed  (pri+sec)/2",
        f"{thfpi_group1_composable_pipeline_result_dict.get('kmeans_synthetic_seed', 0):.10f}")
    _print_labeled_value_row(
        "centroid_0  [PRIMARY]",
        f"{thfpi_group1_composable_pipeline_result_dict.get('kmeans_centroid_0_primary', 0):.10f}  "
        f"[{thfpi_group1_composable_pipeline_result_dict.get('kmeans_centroid_0_primary', 0)*1440:.4f} min]")
    _print_labeled_value_row(
        "centroid_1  [SYNTHETIC]",
        f"{thfpi_group1_composable_pipeline_result_dict.get('kmeans_centroid_1_synthetic_midpoint', 0):.10f}  "
        f"[{thfpi_group1_composable_pipeline_result_dict.get('kmeans_centroid_1_synthetic_midpoint', 0)*1440:.4f} min]  <- NOVEL midpoint")
    _print_labeled_value_row(
        "centroid_2  [SECONDARY]",
        f"{thfpi_group1_composable_pipeline_result_dict.get('kmeans_centroid_2_secondary', 0):.10f}  "
        f"[{thfpi_group1_composable_pipeline_result_dict.get('kmeans_centroid_2_secondary', 0)*1440:.4f} min]")
    _print_labeled_value_row(
        "iteration_count",
        f"{thfpi_group1_composable_pipeline_result_dict.get('kmeans_iteration_count', 0)}  (0 = immediate convergence)")

def _log_augmentation_conjunctive_delta_block(thfpi_group1_composable_pipeline_result_dict):
    _print_subsection_header_box("AUGMENTATION  TC-C  Conjunctive Median Delta")
    diff_0_to_1_day_fraction        = thfpi_group1_composable_pipeline_result_dict.get("delta_diff_0_to_1",          0)
    diff_1_to_2_day_fraction        = thfpi_group1_composable_pipeline_result_dict.get("delta_diff_1_to_2",          0)
    conjunctive_median_day_fraction = thfpi_group1_composable_pipeline_result_dict.get("delta_conjunctive_median",   0)
    extrapolated_delta_day_fraction = thfpi_group1_composable_pipeline_result_dict.get("delta_extrapolated",         0)
    raw_inter_probe_gap_day_fraction = abs(
        thfpi_group1_composable_pipeline_result_dict.get("kmeans_centroid_2_secondary", 0) -
        thfpi_group1_composable_pipeline_result_dict.get("kmeans_centroid_0_primary",   0))
    _print_labeled_value_row("diff_0_to_1  (c1 - c0)",
        f"{diff_0_to_1_day_fraction:.10f}  [{diff_0_to_1_day_fraction*1440:.4f} min]")
    _print_labeled_value_row("diff_1_to_2  (c2 - c1)",
        f"{diff_1_to_2_day_fraction:.10f}  [{diff_1_to_2_day_fraction*1440:.4f} min]")
    _print_labeled_value_row("conjunctive_median  NOVEL",
        f"{conjunctive_median_day_fraction:.10f}  [{conjunctive_median_day_fraction*1440:.4f} min]  median([d01,d12])")
    _print_labeled_value_row("extrapolated_delta",
        f"{extrapolated_delta_day_fraction:.10f}  [{extrapolated_delta_day_fraction*1440:.4f} min]")
    _print_labeled_value_row("raw_gap  |c2-c0|",
        f"{raw_inter_probe_gap_day_fraction:.10f}  [{raw_inter_probe_gap_day_fraction*1440:.4f} min]")
    conjunctive_vs_raw_ratio = (conjunctive_median_day_fraction / raw_inter_probe_gap_day_fraction
                                if raw_inter_probe_gap_day_fraction > 0 else 0)
    _print_labeled_value_row("conjunctive_vs_raw ratio",
        f"{conjunctive_vs_raw_ratio:.6f}  (always 0.5 for symmetric k=3)")

def _log_augmentation_velocity_block(thfpi_group1_consolidated_31key_result_dict):
    _print_subsection_header_box("AUGMENTATION  TC-V  Synthetic Centroid Tension Velocity")
    _print_labeled_value_row(
        "synthetic_position",
        f"{thfpi_group1_consolidated_31key_result_dict.get('synthetic_centroid_position', 0):.10f}")
    real_centroid_count = thfpi_group1_consolidated_31key_result_dict.get("real_centroid_count", 0)
    _print_labeled_value_row("real_centroid_count  (N)", str(real_centroid_count))
    signed_tension_vector_list = thfpi_group1_consolidated_31key_result_dict.get("per_centroid_signed_tension_vectors", [])
    squared_tension_value_list = thfpi_group1_consolidated_31key_result_dict.get("per_centroid_squared_tension_values",  [])
    for tension_index, (signed_tension_value, squared_tension_value) in enumerate(
            zip(signed_tension_vector_list, squared_tension_value_list)):
        _print_labeled_value_row(
            f"  signed_tension[{tension_index}]",
            f"{signed_tension_value:.10f}  [{signed_tension_value*1440:.4f} min]")
        _print_labeled_value_row(
            f"  squared_tension[{tension_index}]",
            f"{squared_tension_value:.16f}")
    sum_of_all_squared_tensions                  = thfpi_group1_consolidated_31key_result_dict.get("sum_of_all_squared_tensions",              0)
    cumulative_magnitude_velocity_day_fractions  = thfpi_group1_consolidated_31key_result_dict.get("cumulative_magnitude_velocity",            0)
    velocity_in_minutes_value                    = thfpi_group1_consolidated_31key_result_dict.get("velocity_in_minutes",                      0)
    tension_symmetry_indicator_value             = thfpi_group1_consolidated_31key_result_dict.get("tension_symmetry_indicator",               0)
    mean_absolute_tension_value                  = thfpi_group1_consolidated_31key_result_dict.get("mean_absolute_tension",                    0)
    velocity_normalized_value                    = thfpi_group1_consolidated_31key_result_dict.get("velocity_normalized",                      0)
    velocity_normalized_sqrt_n_reference_value   = thfpi_group1_consolidated_31key_result_dict.get("velocity_normalized_sqrt_n_reference",     0)
    velocity_normalized_deviation_from_sqrt_n_value = thfpi_group1_consolidated_31key_result_dict.get("velocity_normalized_deviation_from_sqrt_n", 0)
    _print_labeled_value_row("sum_of_squared_tensions",       f"{sum_of_all_squared_tensions:.16f}")
    _print_labeled_value_row("cumulative_magnitude  sqrt(St2)",
        f"{cumulative_magnitude_velocity_day_fractions:.10f}  [{velocity_in_minutes_value:.6f} min]  <- ISO period driver")
    _print_labeled_value_row("velocity_in_minutes",           f"{velocity_in_minutes_value:.6f} min")
    _print_labeled_value_row("tension_symmetry_indicator",    f"{tension_symmetry_indicator_value:.10f}  (0 = perfect symmetry)")
    _print_labeled_value_row("mean_absolute_tension",         f"{mean_absolute_tension_value:.10f}")
    _print_labeled_value_row("velocity_normalized",           f"{velocity_normalized_value:.10f}  <- ISO C(t) driver")
    _print_labeled_value_row("velocity_norm_sqrt_N_ref",      f"{velocity_normalized_sqrt_n_reference_value:.10f}")
    _print_labeled_value_row("velocity_norm_deviation",       f"{velocity_normalized_deviation_from_sqrt_n_value:.10f}")
    _print_labeled_value_row("decimal_velocity_50fig",
        thfpi_group1_consolidated_31key_result_dict.get("decimal_velocity_50fig", ""))
    _print_labeled_value_row("decimal_normalized_50fig",
        thfpi_group1_consolidated_31key_result_dict.get("decimal_normalized_50fig", ""))

def _log_augmentation_all_blocks(thfpi_group1_composable_14key_result_dict):
    _log_augmentation_normalization_block(thfpi_group1_composable_14key_result_dict)
    _log_augmentation_kmeans_block(thfpi_group1_composable_14key_result_dict)
    _log_augmentation_conjunctive_delta_block(thfpi_group1_composable_14key_result_dict)

def _log_iso_scalar_propagation_chain(
        velocity_normalized_iso_c_driver_float,
        cumulative_magnitude_iso_period_driver_float,
        iso_central_value_C_float,
        iso_n_connected_integer,
        iso_vertex_weights_list,
        pipeline_wall_clock_start_ns):
    _print_subsection_header_box("DISTRIBUTION  SCALAR PROPAGATION CHAIN  TC-V → ISO")
    _print_labeled_value_row(
        "STEP 1  velocity_normalized  (TC-V OUT-10)",
        f"{velocity_normalized_iso_c_driver_float:.10f}  <- feeds ISO_M * vel_norm + ISO_B")
    _print_labeled_value_row(
        "STEP 2  cumulative_magnitude  (TC-V OUT-06)",
        f"{cumulative_magnitude_iso_period_driver_float:.10f}  <- feeds ISO_BASE * cum_mag = period_seconds")
    _print_labeled_value_row(
        "STEP 3  C = ISO_M * vel_norm + ISO_B",
        f"{IsomorphicDistributionEngine.ISO_M} * {velocity_normalized_iso_c_driver_float:.10f} + "
        f"{IsomorphicDistributionEngine.ISO_B} = {iso_central_value_C_float:.10f}")
    iso_period_seconds_computed = (IsomorphicDistributionEngine.ISO_BASE_PERIOD_SECONDS *
                                   max(cumulative_magnitude_iso_period_driver_float,
                                       IsomorphicDistributionEngine.ISO_MIN_PERIOD_SCALE))
    _print_labeled_value_row(
        "STEP 4  period_seconds = ISO_BASE * cum_mag",
        f"{IsomorphicDistributionEngine.ISO_BASE_PERIOD_SECONDS} * "
        f"{cumulative_magnitude_iso_period_driver_float:.10f} = "
        f"{iso_period_seconds_computed:.10f} s")
    _print_labeled_value_row(
        "STEP 5  N_connected  (8 cube vertices)",
        f"{iso_n_connected_integer}  — C is divided evenly by N before per-node sine weighting")
    uniform_scalar_c_divided_by_n = iso_central_value_C_float / iso_n_connected_integer
    _print_labeled_value_row(
        "STEP 6  uniform scalar per node  C / N",
        f"{iso_central_value_C_float:.10f} / {iso_n_connected_integer} = "
        f"{uniform_scalar_c_divided_by_n:.10f}  <- same for all nodes before w_i")
    print(f"  {'─'*(orchestrator_log_box_width_characters-2)}")
    print(f"    {'nd':<4} {'w_i  (1+sin)':>16} {'C/N':>14} {'V_i = C*w_i/N':>16}  "
          f"{'scale = 1+V_i*0.5':>20}  {'hue = V_i*180':>14}")
    print(f"    {'----':<4} {'----------------':>16} {'----------':>14} {'----------------':>16}  "
          f"{'--------------------':>20}  {'----------':>14}")
    for vertex_index, vertex_sine_weight in enumerate(iso_vertex_weights_list):
        per_node_distribution_value_V_i  = iso_central_value_C_float * vertex_sine_weight / iso_n_connected_integer
        per_node_isometric_scale_factor  = 1.0 + per_node_distribution_value_V_i * 0.5
        per_node_colour_hue_degrees      = per_node_distribution_value_V_i * 180.0
        print(f"    {vertex_index:<4} {vertex_sine_weight:>16.10f} "
              f"{uniform_scalar_c_divided_by_n:>14.10f} "
              f"{per_node_distribution_value_V_i:>16.10f}  "
              f"{per_node_isometric_scale_factor:>20.10f}  "
              f"{per_node_colour_hue_degrees:>14.6f}°")
    print(f"  {'─'*(orchestrator_log_box_width_characters-2)}")
    _print_labeled_value_row(
        "STEP 7  sync runs on tel wall-clock  (stateless — no class timer)",
        "elapsed_seconds passed explicitly by Orchestrator as (perf_counter_ns - tel.start_ns) / 1e9")
    _print_labeled_value_row(
        "STEP 8  triangle_wave(elapsed, period)  -> t in [0,1]",
        "used only as phase clock; does NOT modulate V_i in current implementation")
    _print_labeled_value_row(
        "STEP 9  P_i' = P_i * (1 + V_i * 0.5)  (isometric outward push)",
        "scale factor applied identically to all three coordinates of each vertex")
    _log_wall_clock_timing_boundary("scalar propagation log complete", pipeline_wall_clock_start_ns)

# ── FIX: every numeric getattr coerced through  (getattr(...) or 0)  ─────────
# DynamicSection.__getattr__ returns None for missing attrs; when an attr IS
# present but holds None the default of getattr(obj, attr, 0) is never used.
# (x or 0) collapses both None and 0.0 to 0 safely for all float fields here.
# ─────────────────────────────────────────────────────────────────────────────
def _log_iso_distribution_output(jit_object, matched_groups_list):
    iso_distribution_engine_ref      = IsomorphicDistributionEngine
    iso_central_node_dynamic_section = jit_object.tandem_interface.iso_central

    _print_subsection_header_box("DISTRIBUTION  ISO ENGINE  Central Node")
    if iso_central_node_dynamic_section is None:
        print("    iso_central is None — sync() was not called")
        return

    _print_labeled_value_row("sync_count",
        str(getattr(iso_central_node_dynamic_section, "sync_count", "?")))
    _print_labeled_value_row("elapsed_seconds",
        f"{(getattr(iso_central_node_dynamic_section, 'elapsed_seconds', None) or 0):.6f} s")
    _print_labeled_value_row("period_seconds",
        f"{(getattr(iso_central_node_dynamic_section, 'period_seconds', None) or 0):.6f} s  <- ISO_BASE * cum_mag")
    _print_labeled_value_row("triangle_wave_t",
        f"{(getattr(iso_central_node_dynamic_section, 'triangle_wave_t', None) or 0):.6f}  in [0,1]")
    _print_labeled_value_row("velocity_normalized",
        f"{(getattr(iso_central_node_dynamic_section, 'velocity_normalized', None) or 0):.10f}  <- TC-V output")
    _print_labeled_value_row("cumulative_magnitude",
        f"{(getattr(iso_central_node_dynamic_section, 'cumulative_magnitude', None) or 0):.10f}  <- TC-V output")
    _print_labeled_value_row("central_value_C",
        f"{(getattr(iso_central_node_dynamic_section, 'central_value_C', None) or 0):.6f}  (ISO_M * vel_norm + ISO_B)")
    _print_labeled_value_row("n_connected",
        str(getattr(iso_central_node_dynamic_section, "n_connected", "?")))

    iso_n_connected_count = getattr(iso_central_node_dynamic_section, "n_connected", 8) or 8
    _print_subsection_header_box("DISTRIBUTION  ISO ENGINE  Per-Node Output")
    print(f"    {'nd':<4} {'w_i':>10} {'V_i':>12} {'scale':>10} {'hue°':>8}  "
          f"{'P_x':>8} {'P_y':>8} {'P_z':>8}  "
          f"{'P_x_prime':>10} {'P_y_prime':>10} {'P_z_prime':>10}  matched group")
    print(f"    {'─'*4} {'─'*10} {'─'*12} {'─'*10} {'─'*8}  "
          f"{'─'*8} {'─'*8} {'─'*8}  {'─'*10} {'─'*10} {'─'*10}  ─────────────────────")
    for iso_node_vertex_index in range(iso_n_connected_count):
        iso_node_dynamic_section = getattr(
            jit_object.tandem_interface,
            f"iso_node_{iso_node_vertex_index}", None)
        if iso_node_dynamic_section is None:
            continue
        vertex_sine_weight_value   = (getattr(iso_node_dynamic_section, "w_i",         None) or 0)
        per_node_distribution_V_i  = (getattr(iso_node_dynamic_section, "V_i",         None) or 0)
        isometric_scale_factor     = (getattr(iso_node_dynamic_section, "scale_factor", None) or 0)
        colour_hue_degrees         = (getattr(iso_node_dynamic_section, "hue_degrees",  None) or 0)
        original_position_x        = (getattr(iso_node_dynamic_section, "P_x",         None) or 0)
        original_position_y        = (getattr(iso_node_dynamic_section, "P_y",         None) or 0)
        original_position_z        = (getattr(iso_node_dynamic_section, "P_z",         None) or 0)
        pushed_position_x_prime    = (getattr(iso_node_dynamic_section, "P_x_prime",   None) or 0)
        pushed_position_y_prime    = (getattr(iso_node_dynamic_section, "P_y_prime",   None) or 0)
        pushed_position_z_prime    = (getattr(iso_node_dynamic_section, "P_z_prime",   None) or 0)
        matched_group_label = ""
        if iso_node_vertex_index < len(matched_groups_list):
            matched_group_dict           = matched_groups_list[iso_node_vertex_index]
            primary_event_dict           = matched_group_dict.get("matched_group_primary_probe_contributing_scan_event",   {})
            secondary_event_dict         = matched_group_dict.get("matched_group_secondary_probe_contributing_scan_event", {})
            matched_group_time_gap_hours = (matched_group_dict.get("matched_group_actual_time_gap_between_probe_events_hours", 0) or 0)
            matched_group_label = (
                f"{iso_node_vertex_index+1}  "
                f"{primary_event_dict.get('scan_event_physical_location_city_state','?')[:14]} <-> "
                f"{secondary_event_dict.get('scan_event_physical_location_city_state','?')[:14]}  "
                f"{matched_group_time_gap_hours:.4f}h")
        print(f"    {iso_node_vertex_index:<4} "
              f"{vertex_sine_weight_value:>10.4f} "
              f"{per_node_distribution_V_i:>12.6f} "
              f"{isometric_scale_factor:>10.4f} "
              f"{colour_hue_degrees:>8.2f}  "
              f"{original_position_x:>8.4f} {original_position_y:>8.4f} {original_position_z:>8.4f}  "
              f"{pushed_position_x_prime:>10.4f} {pushed_position_y_prime:>10.4f} {pushed_position_z_prime:>10.4f}  "
              f"{matched_group_label}")
        formula_line = (
            f"w_{iso_node_vertex_index}={vertex_sine_weight_value:.4f}  "
            f"V_{iso_node_vertex_index}=C·{vertex_sine_weight_value:.4f}/8={per_node_distribution_V_i:.4f}  "
            f"scale={isometric_scale_factor:.4f}  hue={colour_hue_degrees:.1f}°  "
            f"P'=[{pushed_position_x_prime:.4f},{pushed_position_y_prime:.4f},{pushed_position_z_prime:.4f}]")
        _print_labeled_value_row(f"  formula[{iso_node_vertex_index}]", formula_line)

    iso_telemetry_log_list = getattr(jit_object.tandem_interface, "iso_telemetry_log", []) or []
    _print_subsection_header_box("DISTRIBUTION  ISO ENGINE  Telemetry Log")
    _print_labeled_value_row("total sync entries in iso_telemetry_log", str(len(iso_telemetry_log_list)))
    if iso_telemetry_log_list:
        most_recent_telemetry_entry_dict = iso_telemetry_log_list[-1]
        _print_labeled_value_row("last_entry sync_count",
            str(most_recent_telemetry_entry_dict.get("sync_count", "?")))
        _print_labeled_value_row("last_entry elapsed_seconds",
            f"{(most_recent_telemetry_entry_dict.get('elapsed_seconds', None) or 0):.6f} s")
        _print_labeled_value_row("last_entry period_seconds",
            f"{(most_recent_telemetry_entry_dict.get('period_seconds',  None) or 0):.6f} s")
        _print_labeled_value_row("last_entry triangle_wave_t",
            f"{(most_recent_telemetry_entry_dict.get('triangle_wave_t', None) or 0):.6f}")
        _print_labeled_value_row("last_entry central_value_C",
            f"{(most_recent_telemetry_entry_dict.get('central_value_C', None) or 0):.6f}")
        _print_labeled_value_row("last_entry n_connected",
            str(most_recent_telemetry_entry_dict.get("n_connected", "?")))
        _print_labeled_value_row("node_snapshots count",
            str(len(most_recent_telemetry_entry_dict.get("node_snapshots", []) or [])))

    _print_subsection_header_box("DISTRIBUTION  ISO ENGINE  Telemetry Analysis")
    iso_telemetry_analysis_dynamic_section = getattr(
        jit_object.tandem_interface, "iso_telemetry_analysis", None)
    if iso_telemetry_analysis_dynamic_section is None:
        print("    iso_telemetry_analysis not yet computed")
        return
    _print_labeled_value_row("total_sync_cycles",
        str(getattr(iso_telemetry_analysis_dynamic_section, "total_sync_cycles", "?")))
    _print_labeled_value_row("C_min",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'C_min', None) or 0):.6f}")
    _print_labeled_value_row("C_max",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'C_max', None) or 0):.6f}")
    _print_labeled_value_row("C_mean",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'C_mean', None) or 0):.6f}")
    _print_labeled_value_row("period_seconds_min",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'period_seconds_min', None) or 0):.6f} s")
    _print_labeled_value_row("period_seconds_max",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'period_seconds_max', None) or 0):.6f} s")
    _print_labeled_value_row("triangle_wave_t_min",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'triangle_wave_t_min', None) or 0):.6f}")
    _print_labeled_value_row("triangle_wave_t_max",
        f"{(getattr(iso_telemetry_analysis_dynamic_section, 'triangle_wave_t_max', None) or 0):.6f}")


# ─────────────────────────────────────────────────────────────────────────────
# FactoryOrchestrator
# ─────────────────────────────────────────────────────────────────────────────

class FactoryOrchestrator:

    @staticmethod
    def run(primary_probe_input, secondary_probe_input):

        pipeline_wall_clock_start_ns = time.perf_counter_ns()                         # [1]

        _print_section_header_box("THFPI PIPELINE  FactoryOrchestrator.run()  START")
        _print_labeled_value_row("primary_probe_uid",   primary_probe_input.uid)
        _print_labeled_value_row("secondary_probe_uid", secondary_probe_input.uid)
        _log_wall_clock_timing_boundary("FactoryOrchestrator.run() entered", pipeline_wall_clock_start_ns)

        # ── [ORC-M] MANIFEST CONVERSION ──────────────────────────────────────
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  [ORC-M] MANIFEST CONVERSION")
        orc_m_phase_wall_clock_start_ns = time.perf_counter_ns()                       # [2]

        for probe_being_converted in [primary_probe_input, secondary_probe_input]:     # [3]
            existing_manifest_attribute = getattr(
                probe_being_converted.payload, 'package_shipping_manifest', None)      # [4]
            raw_manifest_dict_attribute = getattr(
                probe_being_converted.payload, 'raw_manifest_data', None)              # [5]

            manifest_already_fully_built = (                                           # [6]
                isinstance(existing_manifest_attribute, DynamicSection)
                and getattr(existing_manifest_attribute, 'scan_event_1', None) is not None)

            if manifest_already_fully_built:
                print(f"    probe={probe_being_converted.uid}  [ORC-M-SKIP] "
                      f"package_shipping_manifest already built — "
                      f"scan_event_1 present, raw_manifest_data conversion skipped")
                continue                                                                # [7]

            if raw_manifest_dict_attribute is None:
                print(f"    probe={probe_being_converted.uid}  [ORC-M-SKIP] "
                      f"raw_manifest_data is None and no pre-built manifest — skipping")
                continue                                                                # [8]

            print(f"    probe={probe_being_converted.uid}  [ORC-M-CONVERT] "
                  f"converting raw_manifest_data dict -> DynamicSection")
            probe_being_converted.payload.package_shipping_manifest = DynamicSection()  # [9]
            probe_being_converted.payload.package_shipping_manifest.package_tracking_id = (
                raw_manifest_dict_attribute["package_tracking_id"])                    # [10]

            for raw_scan_event_dict in raw_manifest_dict_attribute["scan_event_history"]:  # [11]
                scan_event_sequence_integer = raw_scan_event_dict["scan_event_sequence_number"]
                scan_event_dynamic_section  = DynamicSection()                         # [12]
                scan_event_dynamic_section.scan_event_sequence_number = scan_event_sequence_integer
                scan_event_dynamic_section.scan_location_city_state   = raw_scan_event_dict["scan_location_city_state"]
                scan_event_dynamic_section.scan_timestamp_utc         = raw_scan_event_dict["scan_timestamp_utc"]   # [13]
                scan_event_dynamic_section.scan_event_id              = raw_scan_event_dict["scan_event_id"]
                setattr(probe_being_converted.payload.package_shipping_manifest,
                        f"scan_event_{scan_event_sequence_integer}",
                        scan_event_dynamic_section)                                    # [14]
                print(f"      scan_event_{scan_event_sequence_integer}  "
                      f"{raw_scan_event_dict['scan_location_city_state']}  "
                      f"{raw_scan_event_dict['scan_timestamp_utc']}")

        _log_wall_clock_timing_boundary(
            "[ORC-M] complete", pipeline_wall_clock_start_ns, orc_m_phase_wall_clock_start_ns)  # [15]

        # ── [ORC-T] TEMPORAL MATCHING ─────────────────────────────────────────
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  [ORC-T] TEMPORAL MATCHING")
        orc_t_phase_wall_clock_start_ns = time.perf_counter_ns()                       # [16]

        cross_probe_temporal_match_tolerance_hours = (                                 # [17]
            ScanEventTemporalMatcher.SCAN_EVENT_CROSS_PROBE_TEMPORAL_MATCH_TOLERANCE_HOURS)
        _print_labeled_value_row("tolerance_hours", str(cross_probe_temporal_match_tolerance_hours))

        primary_probe_chronological_scan_event_list = (                                # [18]
            ScanEventTemporalMatcher
            .extract_chronologically_ordered_scan_event_list_from_probe_manifest(
                primary_probe_input.payload.package_shipping_manifest))

        secondary_probe_chronological_scan_event_list = (                              # [19]
            ScanEventTemporalMatcher
            .extract_chronologically_ordered_scan_event_list_from_probe_manifest(
                secondary_probe_input.payload.package_shipping_manifest))

        _print_labeled_value_row("primary_events_extracted",   str(len(primary_probe_chronological_scan_event_list)))
        _print_labeled_value_row("secondary_events_extracted", str(len(secondary_probe_chronological_scan_event_list)))

        matched_groups_list, primary_probe_unmatched_event_list, secondary_probe_unmatched_event_list = (  # [20]
            ScanEventTemporalMatcher
            .match_and_group_cross_probe_scan_events_by_time_period_window(
                primary_probe_chronological_scan_event_list,
                secondary_probe_chronological_scan_event_list,
                cross_probe_temporal_match_tolerance_hours))

        _print_labeled_value_row("matched_groups_count",  str(len(matched_groups_list)))
        _print_labeled_value_row("pri_unmatched_count",   str(len(primary_probe_unmatched_event_list)))
        _print_labeled_value_row("sec_unmatched_count",   str(len(secondary_probe_unmatched_event_list)))

        for matched_group_summary_dict in matched_groups_list:                         # [21]
            matched_group_sequence_number      = matched_group_summary_dict["matched_group_sequential_number"]
            matched_group_time_gap_hours       = matched_group_summary_dict["matched_group_actual_time_gap_between_probe_events_hours"]
            matched_group_primary_event_dict   = matched_group_summary_dict["matched_group_primary_probe_contributing_scan_event"]
            matched_group_secondary_event_dict = matched_group_summary_dict["matched_group_secondary_probe_contributing_scan_event"]
            print(f"    grp={matched_group_sequence_number:>2}  gap={matched_group_time_gap_hours:.4f}h  "
                  f"PRI:{matched_group_primary_event_dict['scan_event_physical_location_city_state']:<22} "
                  f"{matched_group_primary_event_dict['scan_event_timestamp_utc_string']}  "
                  f"SEC:{matched_group_secondary_event_dict['scan_event_physical_location_city_state']:<22} "
                  f"{matched_group_secondary_event_dict['scan_event_timestamp_utc_string']}")

        _log_wall_clock_timing_boundary(
            "[ORC-T] complete", pipeline_wall_clock_start_ns, orc_t_phase_wall_clock_start_ns)  # [22]

        # ── [ORC-T] WRITE MATCHED GROUPS → tandem_interface ──────────────────
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  [ORC-T] WRITING MATCHED GROUPS -> tandem_interface")
        orc_t_write_phase_wall_clock_start_ns = time.perf_counter_ns()                # [23]

        jit_accumulator_object = JITObject()                                           # [24]
        jit_accumulator_object.tandem_interface.matched_group_count = len(matched_groups_list)  # [25]

        for matched_group_write_dict in matched_groups_list:                           # [26]
            write_group_sequence_number = matched_group_write_dict["matched_group_sequential_number"]   # [27]
            matched_group_slot_section  = DynamicSection()                             # [28]

            matched_group_slot_section.matched_group_sequential_number = write_group_sequence_number  # [29]
            matched_group_slot_section.temporal_match_tolerance_hours_used_to_form_this_group = (
                matched_group_write_dict["temporal_match_tolerance_hours_used_to_form_this_group"])   # [30]
            matched_group_slot_section.matched_group_actual_time_gap_between_probe_events_hours = (
                matched_group_write_dict["matched_group_actual_time_gap_between_probe_events_hours"]) # [31]

            write_primary_event_dict   = matched_group_write_dict["matched_group_primary_probe_contributing_scan_event"]    # [32]
            write_secondary_event_dict = matched_group_write_dict["matched_group_secondary_probe_contributing_scan_event"]  # [33]

            matched_group_slot_section.matched_group_primary_scan_event_sequence_number                = write_primary_event_dict["scan_event_sequence_number"]
            matched_group_slot_section.matched_group_primary_scan_event_physical_location_city_state   = write_primary_event_dict["scan_event_physical_location_city_state"]
            matched_group_slot_section.matched_group_primary_scan_event_timestamp_utc_string           = write_primary_event_dict["scan_event_timestamp_utc_string"]
            matched_group_slot_section.matched_group_primary_scan_event_id                             = write_primary_event_dict["scan_event_id"]
            matched_group_slot_section.matched_group_secondary_scan_event_sequence_number              = write_secondary_event_dict["scan_event_sequence_number"]
            matched_group_slot_section.matched_group_secondary_scan_event_physical_location_city_state = write_secondary_event_dict["scan_event_physical_location_city_state"]
            matched_group_slot_section.matched_group_secondary_scan_event_timestamp_utc_string         = write_secondary_event_dict["scan_event_timestamp_utc_string"]
            matched_group_slot_section.matched_group_secondary_scan_event_id                           = write_secondary_event_dict["scan_event_id"]

            setattr(jit_accumulator_object.tandem_interface,
                    f"hf_input_group_{write_group_sequence_number}",
                    matched_group_slot_section)                                        # [34]
            print(f"    hf_input_group_{write_group_sequence_number:<2} written  "
                  f"PRI_seq={write_primary_event_dict['scan_event_sequence_number']}  "
                  f"SEC_seq={write_secondary_event_dict['scan_event_sequence_number']}  "
                  f"gap={matched_group_write_dict['matched_group_actual_time_gap_between_probe_events_hours']:.4f}h")

        _log_wall_clock_timing_boundary(
            "tandem_interface matched groups written",
            pipeline_wall_clock_start_ns,
            orc_t_write_phase_wall_clock_start_ns)                                     # [35]

        # ── PHASE SEQUENCING ──────────────────────────────────────────────────
        pipeline_telemetry_object = PipelineTelemetry()                                # [36]

        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  PHASE SEQUENCING")

        phase_1_wall_clock_start_ns = time.perf_counter_ns()                          # [37]
        _log_wall_clock_timing_boundary(
            "Factory_Phase_1  ENTER", pipeline_wall_clock_start_ns, phase_1_wall_clock_start_ns)
        jit_accumulator_object, primary_probe_input, secondary_probe_input = (
            Factory_Phase_1.execute(
                jit_accumulator_object, primary_probe_input, secondary_probe_input, pipeline_telemetry_object))  # [38]
        _log_wall_clock_timing_boundary(
            "Factory_Phase_1  EXIT", pipeline_wall_clock_start_ns, phase_1_wall_clock_start_ns)  # [39]

        phase_23_parallel_wall_clock_start_ns = time.perf_counter_ns()                # [40]
        _log_wall_clock_timing_boundary(
            "Factory_Phase_2 + Factory_Phase_3  PARALLEL START",
            pipeline_wall_clock_start_ns, phase_23_parallel_wall_clock_start_ns)
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as thread_pool_executor:   # [41]
            phase_2_future_result = thread_pool_executor.submit(
                Factory_Phase_2.execute,
                jit_accumulator_object, primary_probe_input, secondary_probe_input, pipeline_telemetry_object)  # [42]
            phase_3_future_result = thread_pool_executor.submit(
                Factory_Phase_3.execute,
                jit_accumulator_object, primary_probe_input, secondary_probe_input, pipeline_telemetry_object)  # [43]
            phase_2_returned_tuple = phase_2_future_result.result()                   # [44]
            phase_3_returned_tuple = phase_3_future_result.result()                   # [45]
        jit_accumulator_object = phase_2_returned_tuple[0]                            # [46]
        primary_probe_input    = phase_2_returned_tuple[1]                            # [47]
        secondary_probe_input  = phase_2_returned_tuple[2]                            # [48]
        _log_wall_clock_timing_boundary(
            "Factory_Phase_2 + Factory_Phase_3  PARALLEL END",
            pipeline_wall_clock_start_ns, phase_23_parallel_wall_clock_start_ns)     # [49]

        phase_4_wall_clock_start_ns = time.perf_counter_ns()                          # [50]
        _log_wall_clock_timing_boundary(
            "Factory_Phase_4  ENTER", pipeline_wall_clock_start_ns, phase_4_wall_clock_start_ns)
        jit_accumulator_object, primary_probe_input, secondary_probe_input = (
            Factory_Phase_4.execute(
                jit_accumulator_object, primary_probe_input, secondary_probe_input, pipeline_telemetry_object))  # [51]
        _log_wall_clock_timing_boundary(
            "Factory_Phase_4  EXIT", pipeline_wall_clock_start_ns, phase_4_wall_clock_start_ns)  # [52]

        # ◀◀ THFPI ACTIVATION POINT 3/3 — transition fires here
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  <<<  THFPI ACTIVATION POINT 3/3 — HFEngine.fire(Phase_1, transition)")
        activation_point_3_wall_clock_start_ns = time.perf_counter_ns()               # [53]
        HFEngine.fire("Phase_1", "transition",
                      jit_accumulator_object, primary_probe_input, secondary_probe_input,
                      pipeline_telemetry_object)                                       # [54]
        pipeline_telemetry_object.record_hf_event(
            "Phase_1", "transition", HFEngine.last_retrieval["reads"])                 # [55]
        _log_hfengine_fire_event(
            "AP3/3  transition", "Phase_1", "transition",
            HFEngine.last_retrieval["reads"], pipeline_wall_clock_start_ns)            # [56]
        _log_wall_clock_timing_boundary(
            "ACTIVATION POINT 3/3  complete",
            pipeline_wall_clock_start_ns, activation_point_3_wall_clock_start_ns)     # [57]

        # ── KMEANS CENTROIDS + VELOCITY WRITE BLOCK ───────────────────────────
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  KMEANS CENTROIDS + VELOCITY WRITE")

        thfpi_group1_consolidated_31key_dict = jit_accumulator_object.tandem_interface.thfpi_group1_consolidated  # [58]
        thfpi_group1_composable_14key_dict   = jit_accumulator_object.tandem_interface.thfpi_group1               # [59]

        if thfpi_group1_consolidated_31key_dict is not None and isinstance(thfpi_group1_consolidated_31key_dict, dict):
            print(f"    source: thfpi_group1_consolidated  ({len(thfpi_group1_consolidated_31key_dict)} keys)")  # [60]
            jit_accumulator_object.tandem_interface.kmeans_centroids.centroid_0_primary            = thfpi_group1_consolidated_31key_dict.get("kmeans_centroid_0_primary")            # [61]
            jit_accumulator_object.tandem_interface.kmeans_centroids.centroid_1_synthetic_midpoint = thfpi_group1_consolidated_31key_dict.get("kmeans_centroid_1_synthetic_midpoint") # [62]
            jit_accumulator_object.tandem_interface.kmeans_centroids.centroid_2_secondary          = thfpi_group1_consolidated_31key_dict.get("kmeans_centroid_2_secondary")          # [63]
            jit_accumulator_object.tandem_interface.synthetic_centroid_velocity_subcomponent.velocity_normalized        = thfpi_group1_consolidated_31key_dict.get("velocity_normalized")          # [64]
            jit_accumulator_object.tandem_interface.synthetic_centroid_velocity_subcomponent.cumulative_magnitude       = thfpi_group1_consolidated_31key_dict.get("velocity_cumulative_magnitude")  # [65]
            jit_accumulator_object.tandem_interface.synthetic_centroid_velocity_subcomponent.velocity_in_minutes        = thfpi_group1_consolidated_31key_dict.get("velocity_in_minutes")            # [66]
            jit_accumulator_object.tandem_interface.synthetic_centroid_velocity_subcomponent.tension_symmetry_indicator = thfpi_group1_consolidated_31key_dict.get("velocity_tension_symmetry_indicator")  # [67]
            velocity_normalized_iso_c_driver_value       = thfpi_group1_consolidated_31key_dict.get("velocity_normalized",          0.0) or 0.0  # [68]
            cumulative_magnitude_iso_period_driver_value = thfpi_group1_consolidated_31key_dict.get("velocity_cumulative_magnitude", 0.0) or 0.0  # [69]
            _log_augmentation_velocity_block(thfpi_group1_consolidated_31key_dict)    # [70]
        else:
            print(f"    thfpi_group1_consolidated absent — falling back to thfpi_group1 composable result")  # [71]
            if thfpi_group1_composable_14key_dict is not None and isinstance(thfpi_group1_composable_14key_dict, dict):
                print(f"    source: thfpi_group1  ({len(thfpi_group1_composable_14key_dict)} keys)")  # [72]
                jit_accumulator_object.tandem_interface.kmeans_centroids.centroid_0_primary            = thfpi_group1_composable_14key_dict.get("kmeans_centroid_0_primary")            # [73]
                jit_accumulator_object.tandem_interface.kmeans_centroids.centroid_1_synthetic_midpoint = thfpi_group1_composable_14key_dict.get("kmeans_centroid_1_synthetic_midpoint") # [74]
                jit_accumulator_object.tandem_interface.kmeans_centroids.centroid_2_secondary          = thfpi_group1_composable_14key_dict.get("kmeans_centroid_2_secondary")          # [75]
            else:
                print(f"    thfpi_group1 also absent — centroids not written")        # [76]
            velocity_normalized_iso_c_driver_value       = 0.0                        # [77]
            cumulative_magnitude_iso_period_driver_value = 0.0                        # [78]

        augmentation_log_source_dict = (thfpi_group1_composable_14key_dict
                                        if (thfpi_group1_composable_14key_dict is not None
                                            and isinstance(thfpi_group1_composable_14key_dict, dict))
                                        else {})
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  AUGMENTATION LOG  (thfpi_group1 composable pipeline)")
        _log_augmentation_all_blocks(augmentation_log_source_dict)                    # [79]

        _log_wall_clock_timing_boundary("centroids + velocity written", pipeline_wall_clock_start_ns)  # [80]

        # ── ISO SYNC ──────────────────────────────────────────────────────────
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  ISO ENGINE SYNC")
        iso_sync_phase_wall_clock_start_ns = time.perf_counter_ns()                   # [81]

        iso_sync_elapsed_seconds_since_pipeline_start = (                             # [82]
            time.perf_counter_ns() - pipeline_telemetry_object.start_ns) / 1_000_000_000.0

        _print_labeled_value_row("velocity_normalized  (TC-V -> ISO C(t))",
            f"{velocity_normalized_iso_c_driver_value:.10f}")
        _print_labeled_value_row("cumulative_magnitude  (TC-V -> ISO period)",
            f"{cumulative_magnitude_iso_period_driver_value:.10f}")
        _print_labeled_value_row("elapsed_seconds  (tel wall-clock)",
            f"{iso_sync_elapsed_seconds_since_pipeline_start:.6f} s")
        _print_labeled_value_row("ISO_BASE_PERIOD_SECONDS",
            str(IsomorphicDistributionEngine.ISO_BASE_PERIOD_SECONDS))
        _print_labeled_value_row("ISO_M", str(IsomorphicDistributionEngine.ISO_M))
        _print_labeled_value_row("ISO_B", str(IsomorphicDistributionEngine.ISO_B))

        iso_expected_central_value_C_preview = (                                      # [83]
            IsomorphicDistributionEngine.ISO_M * velocity_normalized_iso_c_driver_value
            + IsomorphicDistributionEngine.ISO_B)
        _print_labeled_value_row("expected_C  ISO_M*vel+ISO_B",
            f"{iso_expected_central_value_C_preview:.6f}")

        iso_expected_period_seconds_preview = (                                       # [84]
            IsomorphicDistributionEngine.ISO_BASE_PERIOD_SECONDS *
            max(cumulative_magnitude_iso_period_driver_value,
                IsomorphicDistributionEngine.ISO_MIN_PERIOD_SCALE))
        _print_labeled_value_row("expected_period_seconds",
            f"{iso_expected_period_seconds_preview:.6f} s")

        IsomorphicDistributionEngine.sync(                                            # [85]
            iso_sync_elapsed_seconds_since_pipeline_start,
            velocity_normalized_iso_c_driver_value,
            cumulative_magnitude_iso_period_driver_value,
            jit_accumulator_object,
            matched_groups_list,
            sync_count=1)
        IsomorphicDistributionEngine.analyse_cumulative_magnitude_telemetry(          # [86]
            jit_accumulator_object)

        _log_wall_clock_timing_boundary(
            "IsomorphicDistributionEngine.sync() complete",
            pipeline_wall_clock_start_ns, iso_sync_phase_wall_clock_start_ns)         # [87]

        # ── SCALAR PROPAGATION LOG ────────────────────────────────────────────
        print(f"\n  {orchestrator_log_horizontal_rule_string}")
        print(f"  SCALAR PROPAGATION LOG  (TC-V -> ISO distribution chain)")
        iso_central_node_after_sync  = jit_accumulator_object.tandem_interface.iso_central          # [88]
        iso_actual_central_value_C   = (getattr(iso_central_node_after_sync, "central_value_C", None) or 0.0)   # [89]
        iso_actual_n_connected       = (getattr(iso_central_node_after_sync, "n_connected",     None) or 8)     # [90]
        iso_actual_vertex_weights    = IsomorphicDistributionEngine.vertex_weights(iso_actual_n_connected)       # [91]
        _log_iso_scalar_propagation_chain(
            velocity_normalized_iso_c_driver_value,
            cumulative_magnitude_iso_period_driver_value,
            iso_actual_central_value_C,
            iso_actual_n_connected,
            iso_actual_vertex_weights,
            pipeline_wall_clock_start_ns)                                              # [92]

        _log_iso_distribution_output(jit_accumulator_object, matched_groups_list)     # [93]
        _log_wall_clock_timing_boundary("_log_distribution complete", pipeline_wall_clock_start_ns)  # [94]

        # ── THFPI LIVE SNAPSHOT ───────────────────────────────────────────────
        tandem_interface_reference           = jit_accumulator_object.tandem_interface
        thfpi_group1_dict_for_snapshot       = (tandem_interface_reference.thfpi_group1
                                                if (tandem_interface_reference.thfpi_group1
                                                    and isinstance(tandem_interface_reference.thfpi_group1, dict))
                                                else {})
        tandem_interface_results_section     = tandem_interface_reference.results      # [95]
        iso_central_section_for_snapshot     = tandem_interface_reference.iso_central  # [96]

        snapshot_norm_primary_day_fraction      = thfpi_group1_dict_for_snapshot.get("norm_primary",  0)   # [97]
        snapshot_norm_secondary_day_fraction    = thfpi_group1_dict_for_snapshot.get("norm_secondary", 0)  # [98]
        snapshot_norm_signed_delta_day_fraction = snapshot_norm_primary_day_fraction - snapshot_norm_secondary_day_fraction  # [99]
        snapshot_norm_delta_minutes             = snapshot_norm_signed_delta_day_fraction * 1440              # [100]
        snapshot_day_anchor_utc_string          = thfpi_group1_dict_for_snapshot.get("norm_anchor_utc_string", "")   # [101]
        snapshot_centroid_0_primary             = thfpi_group1_dict_for_snapshot.get("kmeans_centroid_0_primary",            0)  # [102]
        snapshot_centroid_1_synthetic           = thfpi_group1_dict_for_snapshot.get("kmeans_centroid_1_synthetic_midpoint", 0)  # [103]
        snapshot_centroid_2_secondary           = thfpi_group1_dict_for_snapshot.get("kmeans_centroid_2_secondary",          0)  # [104]
        snapshot_kmeans_iteration_count         = thfpi_group1_dict_for_snapshot.get("kmeans_iteration_count", 0)  # [105]
        snapshot_diff_0_to_1_day_fraction       = thfpi_group1_dict_for_snapshot.get("delta_diff_0_to_1",  0)   # [106]
        snapshot_diff_1_to_2_day_fraction       = thfpi_group1_dict_for_snapshot.get("delta_diff_1_to_2",  0)   # [107]
        snapshot_conjunctive_median_day_fraction    = thfpi_group1_dict_for_snapshot.get("delta_conjunctive_median", 0)   # [108]
        snapshot_extrapolated_delta_day_fraction    = thfpi_group1_dict_for_snapshot.get("delta_extrapolated",       0)   # [109]
        snapshot_subtract_c2_minus_c0_day_fraction  = TandemCalc.subtract(                                               # [110]
            snapshot_centroid_2_secondary, snapshot_centroid_0_primary)

        print("\n--- THFPI  TANDEM HIGH-FREQUENCY PROBE INTERFACE  LIVE SNAPSHOT ---")
        print(f"  [ LAYER 1 — PROBE INPUT DATA ]")                                          # THFPI-C-01
        print(f"  group                   1")                                                # THFPI-C-02
        print(f"  tolerance               {cross_probe_temporal_match_tolerance_hours} hr") # THFPI-C-03
        print(f"  matched groups          {len(matched_groups_list)}")                       # THFPI-C-04
        print(f"  primary identity        {primary_probe_input.uid}")                        # THFPI-C-05
        print(f"  secondary identity      {secondary_probe_input.uid}")                      # THFPI-C-06
        print(f"  norm primary            {snapshot_norm_primary_day_fraction:.10f}  (day fraction 08:00 UTC)")    # THFPI-C-07
        print(f"  norm secondary          {snapshot_norm_secondary_day_fraction:.10f}  (day fraction 08:15 UTC)")  # THFPI-C-08
        print(f"  norm delta              {snapshot_norm_signed_delta_day_fraction:.10f}  ({snapshot_norm_delta_minutes:.4f} min)")  # THFPI-C-09
        print(f"  [ LAYER 2 — NOVEL COMPUTED RESULTS ]")                                    # THFPI-C-10
        print(f"  day anchor              {snapshot_day_anchor_utc_string}  (midnight UTC 86400-s window zero-point)")  # THFPI-C-11
        print(f"  k-means iterations      {snapshot_kmeans_iteration_count}  (immediate convergence)")               # THFPI-C-12
        print(f"  centroid_0  [PRIMARY]   {snapshot_centroid_0_primary:.10f}")               # THFPI-C-13
        print(f"  centroid_1  [SYNTHETIC] {snapshot_centroid_1_synthetic:.10f}  (NOVEL midpoint)")  # THFPI-C-14
        print(f"  centroid_2  [SECONDARY] {snapshot_centroid_2_secondary:.10f}")             # THFPI-C-15
        print(f"  diff 0->1               {snapshot_diff_0_to_1_day_fraction:.10f}  "
              f"({snapshot_diff_0_to_1_day_fraction*1440:.4f} min  primary->midpoint)")     # THFPI-C-16
        print(f"  diff 1->2               {snapshot_diff_1_to_2_day_fraction:.10f}  "
              f"({snapshot_diff_1_to_2_day_fraction*1440:.4f} min  midpoint->secondary)")   # THFPI-C-17
        print(f"  conjunctive median      {snapshot_conjunctive_median_day_fraction:.10f}  "
              f"({snapshot_conjunctive_median_day_fraction*1440:.4f} min  NOVEL)")          # THFPI-C-18
        print(f"  extrapolated delta      {snapshot_extrapolated_delta_day_fraction:.10f}  "
              f"({snapshot_extrapolated_delta_day_fraction*1440:.4f} min)")                 # THFPI-C-19
        print(f"  [ LAYER 3 — HF ROUTING RESULTS ]")                                        # THFPI-C-20
        print(f"  chk_entry_1             {getattr(tandem_interface_results_section, 'chk_entry_1', 'n/a')}")  # THFPI-C-21
        print(f"  chk_exit_1              {getattr(tandem_interface_results_section, 'chk_exit_1',  'n/a')}")  # THFPI-C-22
        print(f"  chk_btwn_1              {getattr(tandem_interface_results_section, 'chk_btwn_1',  'n/a')}")  # THFPI-C-23
        print(f"  [ THFPI EMBEDDED FUNCTION — TandemCalc.subtract (live from shared bus) ]")  # THFPI-C-24
        print(f"  subtract(c2, c0)        {snapshot_subtract_c2_minus_c0_day_fraction:.10f}  "
              f"({snapshot_subtract_c2_minus_c0_day_fraction*1440:.6f} min)  <- full inter-probe gap")  # THFPI-C-25
        if iso_central_section_for_snapshot is not None:
            print(f"  [ ISO ENGINE — sync_count={getattr(iso_central_section_for_snapshot,'sync_count','?')}]")  # THFPI-C-26
            print(f"  C (central value)       {(getattr(iso_central_section_for_snapshot,'central_value_C',None) or 0):.6f}  (ISO_M * vel_norm + ISO_B)")  # THFPI-C-27
            print(f"  period_seconds          {(getattr(iso_central_section_for_snapshot,'period_seconds',None) or 0):.6f}")   # THFPI-C-28
            print(f"  triangle_wave_t         {(getattr(iso_central_section_for_snapshot,'triangle_wave_t',None) or 0):.6f}")  # THFPI-C-29
            print(f"  N_connected             {getattr(iso_central_section_for_snapshot,'n_connected','?')}")                   # THFPI-C-30
        print("--------------------------------------------------------------------\n")

        _log_wall_clock_timing_boundary("THFPI LIVE SNAPSHOT printed", pipeline_wall_clock_start_ns)  # [111]

        # ── FINAL TIMING SUMMARY ──────────────────────────────────────────────
        _print_section_header_box("THFPI PIPELINE  TIMING SUMMARY")
        _print_labeled_value_row("total_pipeline_wall_time",
             f"{_compute_elapsed_milliseconds_since(pipeline_wall_clock_start_ns):.3f} ms")
        _print_labeled_value_row("phase_1_duration",
            f"{pipeline_telemetry_object.durs.get('Phase_1', 0) / 1e6:.3f} ms")
        _print_labeled_value_row("phase_2_duration",
            f"{pipeline_telemetry_object.durs.get('Phase_2', 0) / 1e6:.3f} ms")
        _print_labeled_value_row("phase_3_duration",
            f"{pipeline_telemetry_object.durs.get('Phase_3', 0) / 1e6:.3f} ms")
        _print_labeled_value_row("phase_4_duration",
            f"{pipeline_telemetry_object.durs.get('Phase_4', 0) / 1e6:.3f} ms")
        _print_labeled_value_row("phase_1_skew",   str(pipeline_telemetry_object.skews.get("Phase_1",  "?")))
        _print_labeled_value_row("phase_2_skew",   str(pipeline_telemetry_object.skews.get("Phase_2",  "?")))
        _print_labeled_value_row("phase_3_skew",   str(pipeline_telemetry_object.skews.get("Phase_3",  "?")))
        _print_labeled_value_row("phase_4_skew",   str(pipeline_telemetry_object.skews.get("Phase_4",  "?")))
        _print_labeled_value_row("phase_1_impact", str(pipeline_telemetry_object.impacts.get("Phase_1", "?")))
        _print_labeled_value_row("phase_2_impact", str(pipeline_telemetry_object.impacts.get("Phase_2", "?")))
        _print_labeled_value_row("phase_3_impact", str(pipeline_telemetry_object.impacts.get("Phase_3", "?")))
        _print_labeled_value_row("phase_4_impact", str(pipeline_telemetry_object.impacts.get("Phase_4", "?")))
        _print_labeled_value_row("hf_event_log_entries", str(len(pipeline_telemetry_object.hf_event_log)))
        for hf_event_log_entry_dict in pipeline_telemetry_object.hf_event_log:         # [112]
            print(f"    hf_event  node={hf_event_log_entry_dict['node']:<10} "
                  f"moment={hf_event_log_entry_dict['moment']:<12} "
                  f"n_rules={hf_event_log_entry_dict['n_rules']}  "
                  f"fns={hf_event_log_entry_dict['fn_names']}")
        _log_wall_clock_timing_boundary("FactoryOrchestrator.run() complete", pipeline_wall_clock_start_ns)  # [113]

        # ── Arch namespace aliases ────────────────────────────────────────────
        JITPipelineArchitecture.Execution.Orchestrator.FactoryOrchestrator = FactoryOrchestrator  # [114]
        JITPipelineArchitecture.Data.Models.JITObject    = JITObject                              # [115]
        JITPipelineArchitecture.Data.Models.DynamicProbe = DynamicProbe                           # [116]

        return jit_accumulator_object, primary_probe_input, secondary_probe_input      # [117]


JITPipelineArchitecture.Execution.Orchestrator.FactoryOrchestrator = FactoryOrchestrator

In [8]:
# @title 8: INVOCATION — probe construction · manifest injection · pipeline execution · VarTracker summary

import time
import concurrent.futures

Arch  = JITPipelineArchitecture
VT    = VarTracker

# ─────────────────────────────────────────────────────────────────────────────
# CONSTRUCT PROBES
# FIX: construct DynamicProbe directly — Arch.Data.Models.DynamicProbe is now
# assigned in the class body of JITPipelineArchitecture so both forms work,
# but direct construction is used here as the canonical form per the master
# context document (probe simulation section).
# ─────────────────────────────────────────────────────────────────────────────
pri_probe = DynamicProbe(uid="ALPHA_ENTITY-SANDBOX")   # direct — never via Arch stub
sec_probe = DynamicProbe(uid="BETA_ENTITY-SANDBOX")    # direct — never via Arch stub

# ─────────────────────────────────────────────────────────────────────────────
# [R5] calc_value — the 2.0 and 3.0 every phase reads in factory step [5]
# ─────────────────────────────────────────────────────────────────────────────
pri_probe.payload.calc_value = 2.0
sec_probe.payload.calc_value = 3.0

# ─────────────────────────────────────────────────────────────────────────────
# [R1] PRIMARY PROBE — belt conveyor step data
# ─────────────────────────────────────────────────────────────────────────────
pri_probe.payload.step1_id   = "A1X"; pri_probe.payload.step1_time = 0.0
pri_probe.payload.step2_id   = "A2X"; pri_probe.payload.step2_time = 19.0
pri_probe.payload.step3_id   = "A3X"; pri_probe.payload.step3_time = 36.0
pri_probe.payload.step4_id   = "A4X"; pri_probe.payload.step4_time = 55.0

# ─────────────────────────────────────────────────────────────────────────────
# [R2] SECONDARY PROBE — belt conveyor step data
# ─────────────────────────────────────────────────────────────────────────────
sec_probe.payload.step1_id   = "B1Y"; sec_probe.payload.step1_time = 3.0
sec_probe.payload.step2_id   = "B2Y"; sec_probe.payload.step2_time = 32.0
sec_probe.payload.step3_id   = "B3Y"; sec_probe.payload.step3_time = 41.0
sec_probe.payload.step4_id   = "B4Y"; sec_probe.payload.step4_time = 52.0

# ─────────────────────────────────────────────────────────────────────────────
# [R3] PRIMARY PROBE — package manifest
# ─────────────────────────────────────────────────────────────────────────────
pri_probe.payload.package_shipping_manifest = DynamicSection()
pri_probe.payload.package_shipping_manifest.package_tracking_id = "PKG-A102"

def _mksec(ds, n, loc, ts, eid):
    s = DynamicSection()
    s.scan_event_sequence_number = n
    s.scan_location_city_state   = loc
    s.scan_timestamp_utc         = ts
    s.scan_event_id              = eid
    setattr(ds, f"scan_event_{n}", s)

_mksec(pri_probe.payload.package_shipping_manifest, 1, "Los Angeles, CA", "2026-03-01 08:00:00", "SE-A-001")
_mksec(pri_probe.payload.package_shipping_manifest, 2, "Los Angeles, CA", "2026-03-01 08:25:00", "SE-A-002")
_mksec(pri_probe.payload.package_shipping_manifest, 3, "Denver, CO",      "2026-03-02 09:00:00", "SE-A-003")
_mksec(pri_probe.payload.package_shipping_manifest, 4, "Denver, CO",      "2026-03-02 09:40:00", "SE-A-004")
_mksec(pri_probe.payload.package_shipping_manifest, 5, "Kansas City, MO", "2026-03-03 14:00:00", "SE-A-005")
_mksec(pri_probe.payload.package_shipping_manifest, 6, "Kansas City, MO", "2026-03-03 14:30:00", "SE-A-006")
_mksec(pri_probe.payload.package_shipping_manifest, 7, "Columbus, OH",    "2026-03-04 11:00:00", "SE-A-007")
_mksec(pri_probe.payload.package_shipping_manifest, 8, "Columbus, OH",    "2026-03-04 11:45:00", "SE-A-008")

# ─────────────────────────────────────────────────────────────────────────────
# [R4] SECONDARY PROBE — package manifest
# ─────────────────────────────────────────────────────────────────────────────
sec_probe.payload.package_shipping_manifest = DynamicSection()
sec_probe.payload.package_shipping_manifest.package_tracking_id = "PKG-B775"

_mksec(sec_probe.payload.package_shipping_manifest, 1, "Seattle, WA",     "2026-03-01 08:15:00", "SE-B-001")
_mksec(sec_probe.payload.package_shipping_manifest, 2, "Seattle, WA",     "2026-03-01 08:50:00", "SE-B-002")
_mksec(sec_probe.payload.package_shipping_manifest, 3, "Denver, CO",      "2026-03-02 09:20:00", "SE-B-003")
_mksec(sec_probe.payload.package_shipping_manifest, 4, "Denver, CO",      "2026-03-02 10:00:00", "SE-B-004")
_mksec(sec_probe.payload.package_shipping_manifest, 5, "Kansas City, MO", "2026-03-03 14:10:00", "SE-B-005")
_mksec(sec_probe.payload.package_shipping_manifest, 6, "Kansas City, MO", "2026-03-03 14:45:00", "SE-B-006")
_mksec(sec_probe.payload.package_shipping_manifest, 7, "Louisville, KY",  "2026-03-04 11:20:00", "SE-B-007")
_mksec(sec_probe.payload.package_shipping_manifest, 8, "Louisville, KY",  "2026-03-04 12:05:00", "SE-B-008")

# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
jit, pri, sec = FactoryOrchestrator.run(pri_probe, sec_probe)

# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT — graphs · narrative logs · VarTracker summary
# ─────────────────────────────────────────────────────────────────────────────
StoryLog.flush_buffer()
VarTracker.summary()


╔══════════════════════════════════════════════════════════════════════════════════╗
║  THFPI PIPELINE  FactoryOrchestrator.run()  START                                  ║
╚══════════════════════════════════════════════════════════════════════════════════╝
    primary_probe_uid                             ALPHA_ENTITY-SANDBOX
    secondary_probe_uid                           BETA_ENTITY-SANDBOX
  ⏱  FactoryOrchestrator.run() entered                     T+    0.426ms

  ────────────────────────────────────────────────────────────────────────────────────
  [ORC-M] MANIFEST CONVERSION
    probe=ALPHA_ENTITY-SANDBOX  [ORC-M-SKIP] package_shipping_manifest already built — scan_event_1 present, raw_manifest_data conversion skipped
    probe=BETA_ENTITY-SANDBOX  [ORC-M-SKIP] package_shipping_manifest already built — scan_event_1 present, raw_manifest_data conversion skipped
  ⏱  [ORC-M] complete                                      T+    0.481ms  (phase Δ 0.026ms)

  ────────────────────────